# Agentic AI Systems — Worklet 15
# Executive Decision Support System
## Lab 2 · Tool-Using Agent — Institutional Data & KPI Computation

**Phase A · Foundations** &nbsp;|&nbsp; **Goal:** give the agent tools it *must* call to act, rather than answering from memory.

---

### What Lab 2 contributes to Worklet 15

| | |
|---|---|
| **Inputs** | Institutional KPIs & dashboards; department reports; parameters. |
| **Outputs** | Verified computed values and parsed inputs. |
| **Agents / nodes** | Coordinator + **tool layer** (the specialist agents arrive in Lab 7). |
| **Deliverable** | An agent that reads institutional KPIs & dashboards and computes/validates through typed tools. |
| **Acceptance** | Key numbers and lookups **always** go through tools, never hand-done in the prompt. |
| **Pitfall to avoid** | Never assert a fact in the prompt that a tool should verify. |

> Chapter 15's Lab 2 pseudocode:
> ```
> a = tool.read('institutional KPIs & dashboards')
> b = tool.read('department reports')
> v = tool.compute(a, b)
> assert v.valid
> ```

### Where Lab 1 left off

Lab 1 built the coordinator loop: it clarifies an underspecified request, calls the data-scope register to learn what it is allowed to read and how final that data is, and produces an editable plan that carries every data caveat it observed. It never touched a number.

This notebook gives it the tools to do so — and then makes it impossible for a figure to reach an answer without having come out of one.

### The four things this notebook is really about

1. **A tool is a contract.** Every tool declares a typed input schema and returns either a result or an explicit `{"error": ...}` — never `None`, never a silent empty list.
2. **One choke point.** `ToolRouter.call()` validates the arguments *before* the call, executes, logs tool + reason + input + output, and raises on error. Nothing calls a tool directly.
3. **The formula lives in the registry, not in the code.** `compute_kpi` reads the approved KPI definition and applies it. An agent cannot invent its own arithmetic.
4. **Grounding is enforced, not requested.** Every answer is scanned; a numeric token that does not appear in a tool output causes the answer to be rejected.

### How to run

Runs **with or without a Gemini API key** — identical to Lab 1, and Cell 2 degrades rather than raising if the API is unavailable for any reason. With a working key, Cell 13 additionally hands the tool schemas to Gemini as function declarations and lets the model choose the calls.

Run top to bottom. Cell 14 is a regression suite; Cell 15 executes the rubric as assertions.

---

### Cell 1 · Environment

In [1]:
# =============================================================================
# Cell 1  ·  Environment setup
# =============================================================================
# Installs the Gemini SDK. Safe to re-run; skipped automatically if present.
try:
    import google.genai  # noqa: F401
    print("google-genai already available.")
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "google-genai"], check=False)
    print("google-genai installed.")


google-genai already available.


### Cell 2 · The LLM backend (unchanged from Lab 1)

Same single interface, same two backends, same prompt log. In this lab the model's job is narrower: it never selects a figure and never does arithmetic — at most it rewords an answer whose numbers were already fixed by the tools, and even that pass is checked before it is accepted.

In [2]:
# =============================================================================
# Cell 2  ·  LLM backend, with a deterministic fallback that CANNOT fail
# =============================================================================
# The agent talks to exactly one interface: LLM.json(...) -> dict.
#
#   * Gemini when the key works  -> temperature 0, response_mime_type=
#     "application/json" + a response_schema, so the model is *forced* to return
#     the exact structure the agent code expects.
#   * OfflineBrain otherwise     -> a rule-based stand-in returning the same
#     structures.
#
# THIS CELL NEVER RAISES, AND NEITHER DOES ANY CALL BELOW IT.
# A missing key, a retired model id, a denied project, a rate limit or an outage
# all degrade to the offline brain with a printed explanation. The control
# structure is what these labs grade, and it is identical on both paths, so a
# degraded run is still a complete, correct, fully-graded run.
#
# Three layers of protection:
#   1. FORCE_OFFLINE       — skip Gemini entirely if you want a guaranteed run.
#   2. Preflight probe     — one tiny call at startup finds the first model id
#                            this key can actually reach, so a bad id surfaces
#                            HERE with a clear message, not 200 lines later.
#   3. Degrade-on-failure  — if Gemini breaks mid-run, that call and every call
#                            after it transparently uses the offline brain.

import os, re, json, time, textwrap
from typing import Any, Dict, List, Optional

# Set to True to bypass Gemini completely and run purely offline.
FORCE_OFFLINE = False

MODEL_CANDIDATES = [
    "gemini-3.6-flash",     # current default for new API keys
    "gemini-flash-latest",  # rolling alias for the newest flash
    "gemini-2.5-flash",     # older keys may still have access
    "gemini-2.0-flash",
]


def _load_api_key() -> Optional[str]:
    """Colab secret -> environment variable -> None (offline mode)."""
    try:
        from google.colab import userdata  # type: ignore
        key = userdata.get("GEMINI_API_KEY")
        if key:
            return key
    except Exception:
        pass
    return os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")


class GeminiBackend:
    """Deterministic, schema-constrained wrapper with model fall-forward."""

    name = "gemini"
    # Any of these means "this model id is not usable on this key" -> try the
    # next candidate. 403/PERMISSION_DENIED belongs here as much as 404 does:
    # both mean the call will never succeed, so retrying is pointless.
    _MODEL_ERROR_MARKERS = ("NOT_FOUND", "404", "403", "PERMISSION_DENIED",
                            "no longer available", "is not found", "not supported",
                            "denied access", "API_KEY_INVALID", "INVALID_ARGUMENT",
                            "quota", "RESOURCE_EXHAUSTED", "429")

    def __init__(self, api_key: str, candidates: List[str] = None):
        from google import genai
        from google.genai import types
        self._types = types
        self.client = genai.Client(api_key=api_key)
        self.candidates = list(candidates or MODEL_CANDIDATES)
        self.model = self.candidates[0]
        self.model_switches: List[str] = []
        self.probe_errors: List[tuple] = []

    # ---- introspection --------------------------------------------------
    def available_models(self) -> List[str]:
        try:
            return sorted(
                m.name.replace("models/", "") for m in self.client.models.list()
                if "generateContent" in (getattr(m, "supported_actions", None) or []))
        except Exception as exc:
            return [f"(could not list models: {type(exc).__name__}: {exc})"]

    # ---- robust JSON extraction -----------------------------------------
    def _extract_json(self, resp) -> Dict[str, Any]:
        """Newer models emit reasoning parts; take only the real text parts."""
        parsed = getattr(resp, "parsed", None)
        if isinstance(parsed, dict) and parsed:
            return parsed
        chunks = []
        for cand in (getattr(resp, "candidates", None) or []):
            content = getattr(cand, "content", None)
            for part in (getattr(content, "parts", None) or []):
                if getattr(part, "thought", False):      # skip reasoning parts
                    continue
                if getattr(part, "text", None):
                    chunks.append(part.text)
        text = ("".join(chunks) or getattr(resp, "text", "") or "").strip()
        if text.startswith("```"):                        # strip a stray fence
            text = re.sub(r"^```[a-zA-Z]*\s*", "", text)
            text = re.sub(r"\s*```$", "", text)
        if not text:
            raise ValueError("model returned no text content")
        return json.loads(text)

    def _config(self, system: str, schema: Dict[str, Any]):
        types = self._types
        return types.GenerateContentConfig(
            system_instruction=system,
            temperature=0.0,                     # deterministic -> reproducible marks
            seed=7,
            response_mime_type="application/json",
            response_schema=schema,
            # We drive the loop ourselves; automatic function calling would hide
            # the control flow these labs are graded on.
            automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
        )

    # ---- preflight: find a model this key can actually call --------------
    def preflight(self) -> Optional[str]:
        probe_schema = {"type": "object",
                        "properties": {"ok": {"type": "boolean"}},
                        "required": ["ok"]}
        while self.candidates:
            candidate = self.candidates[0]
            try:
                resp = self.client.models.generate_content(
                    model=candidate, contents='Reply with exactly {"ok": true}',
                    config=self._config("Reply only with JSON.", probe_schema))
                self._extract_json(resp)
                self.model = candidate
                return candidate
            except Exception as exc:
                self.probe_errors.append((candidate, f"{type(exc).__name__}: "
                                                     f"{str(exc)[:220]}"))
                self.candidates.pop(0)
        return None

    # ---- normal calls ----------------------------------------------------
    def json(self, system: str, prompt: str, schema: Dict[str, Any],
             retries: int = 3) -> Dict[str, Any]:
        cfg = self._config(system, schema)
        last_err, attempts = None, 0
        while attempts < retries:
            try:
                resp = self.client.models.generate_content(
                    model=self.model, contents=prompt, config=cfg)
                return self._extract_json(resp)
            except Exception as exc:
                last_err = exc
                fatal_for_model = any(m in str(exc) for m in self._MODEL_ERROR_MARKERS)
                if fatal_for_model and len(self.candidates) > 1:
                    retired = self.candidates.pop(0)
                    self.model = self.candidates[0]
                    self.model_switches.append(f"{retired} -> {self.model}")
                    print(f"  [model] '{retired}' unusable on this key; "
                          f"switching to '{self.model}'")
                    continue                     # a switch does not burn a retry
                if fatal_for_model:
                    break                        # no candidates left; stop retrying
                attempts += 1
                if attempts < retries:
                    time.sleep(1.5 * attempts)   # transient 503 etc.
        raise RuntimeError(f"model '{self.model}': {last_err}")


class OfflineBrain:
    """Rule-based stand-in for the LLM.

    Mirrors the reference-implementation style used throughout the lab guide:
    the *control structure* is what these labs grade, so it must remain runnable
    and inspectable with no API key. Handlers are registered by task name.
    """

    name = "offline"

    def __init__(self):
        self.handlers = {}

    def register(self, task: str):
        def deco(fn):
            self.handlers[task] = fn
            return fn
        return deco

    def json(self, system: str, prompt: str, schema: Dict[str, Any],
             task: str = "", payload: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
        if task not in self.handlers:
            raise RuntimeError(f"OfflineBrain has no handler for task '{task}'")
        return self.handlers[task](payload or {})


def _explain_gemini_failure(errors: List[tuple]) -> None:
    print("  Gemini could not be used. What each model id returned:")
    for model, err in errors:
        print(f"    - {model:<22} {err}")
    blob = " ".join(e for _, e in errors)
    print("\n  Most likely cause, based on those messages:")
    if "PERMISSION_DENIED" in blob or "denied access" in blob:
        print("    403 PERMISSION_DENIED — the key authenticated, but the Google Cloud")
        print("    project behind it is not allowed to call the Generative Language API.")
        print("    Usual fixes, in order of how often they work:")
        print("      1. In AI Studio (aistudio.google.com/apikey), delete this key and")
        print("         choose 'Create API key in a NEW project'. A key inherited from an")
        print("         existing/organisation project is the common cause.")
        print("      2. If you are on a college/organisation Google account, use a personal")
        print("         @gmail.com account instead — org policy often blocks this API.")
        print("      3. A brand-new key can take a few minutes to activate; retry shortly.")
    elif "API_KEY_INVALID" in blob or "INVALID_ARGUMENT" in blob:
        print("    The key itself was rejected. Check for a stray space when pasting it")
        print("    into the Colab secret, and that the secret is named GEMINI_API_KEY.")
    elif "RESOURCE_EXHAUSTED" in blob or "429" in blob or "quota" in blob:
        print("    Free-tier quota is exhausted for now. It resets; try again later.")
    else:
        print("    Every candidate model id was rejected. Run llm.available_models()")
        print("    and put a working id at the front of MODEL_CANDIDATES.")
    print("\n  THIS DOES NOT BLOCK THE LAB. The notebook has switched to the offline")
    print("  brain and every cell below runs normally — same loop, same tools, same")
    print("  checks, same marks. Only the phrasing is rule-based instead of generated.")


class LLM:
    """Router: Gemini when it works, OfflineBrain otherwise. Never raises."""

    def __init__(self):
        self.offline = OfflineBrain()
        self.backend = None
        self.degraded_reason: Optional[str] = None
        self.calls: List[Dict[str, Any]] = []      # the prompt log (a deliverable)

        if FORCE_OFFLINE:
            self.degraded_reason = "FORCE_OFFLINE is set to True in this cell"
            return
        key = _load_api_key()
        if not key:
            self.degraded_reason = "no GEMINI_API_KEY found in Colab secrets or environment"
            return
        try:
            candidate_backend = GeminiBackend(key)
            print("  Probing which Gemini model this key can reach...")
            if candidate_backend.preflight():
                self.backend = candidate_backend
            else:
                self.degraded_reason = "no candidate model was callable with this key"
                _explain_gemini_failure(candidate_backend.probe_errors)
        except Exception as exc:
            self.degraded_reason = f"{type(exc).__name__}: {exc}"
            _explain_gemini_failure([("client init", str(exc)[:220])])

    @property
    def mode(self) -> str:
        return "gemini" if self.backend else "offline"

    @property
    def model_label(self) -> str:
        return self.backend.model if self.backend else "rule-based OfflineBrain"

    def available_models(self) -> List[str]:
        if not self.backend:
            return ["(offline mode — no working Gemini backend)"]
        return self.backend.available_models()

    def json(self, *, task: str, system: str, prompt: str,
             schema: Dict[str, Any], payload: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
        source = "offline"
        if self.backend:
            try:
                out = self.backend.json(system, prompt, schema)
                source = f"gemini:{self.backend.model}"
            except Exception as exc:
                # Degrade permanently rather than raise: one API problem must not
                # take down a notebook that is being graded.
                self.degraded_reason = str(exc)
                self.backend = None
                print(f"\n  [degraded] Gemini failed on task '{task}': {exc}")
                print("  [degraded] Switching to the offline brain for this and all")
                print("  [degraded] later calls. Every cell below still runs and still")
                print("  [degraded] passes its checks.\n")
                out = self.offline.json(system, prompt, schema, task=task, payload=payload)
        else:
            out = self.offline.json(system, prompt, schema, task=task, payload=payload)
        self.calls.append({
            "call": len(self.calls) + 1, "task": task, "backend": source,
            "system": system.strip(), "prompt": prompt.strip(), "response": out,
        })
        return out


llm = LLM()
print(f"\nLLM backend  : {llm.mode}")
print(f"Model        : {llm.model_label}")
if llm.mode == "gemini":
    print("Verified     : a live probe call succeeded on this model id.")
    print(f"Fallback order remaining: {llm.backend.candidates}")
else:
    print(f"Reason       : {llm.degraded_reason}")
    print("Impact       : none on grading — the loop, the tools, the logs and every")
    print("               acceptance check below are backend-independent.")
print("Both backends return the same JSON structures, so every cell below runs either way.")



LLM backend  : offline
Model        : rule-based OfflineBrain
Reason       : no GEMINI_API_KEY found in Colab secrets or environment
Impact       : none on grading — the loop, the tools, the logs and every
               acceptance check below are backend-independent.
Both backends return the same JSON structures, so every cell below runs either way.


### Cell 3 · The institutional data

Three sources — exactly the three Chapter 15 puts behind the connector: **department reports**, **institutional KPIs & dashboards** (the approved definitions and targets), and **external benchmarks**.

Two deliberate properties of this dataset, because both drive graded behaviour later:

* **The Finance figures are provisional.** `KPI-F1` carries a `figure_status` of `provisional` and a caveat. The Finance Agent guardrail — *never project a forecast as an actual* — has to be enforceable, so the status travels with the figure.
* **The benchmark API is deliberately partial.** Only 3 of 9 KPIs have a published peer median. The other 6 must produce an honest "we don't have that", not an estimate.

In [3]:
# =============================================================================
# Cell 3  ·  The institutional data — what the tools actually read
# =============================================================================
# Three sources, exactly the three Chapter 15 puts behind the connector:
#
#   department reports              -> DEPARTMENT_REPORTS_CSV  (8 departments)
#   institutional KPIs & dashboards -> KPI_DEFINITIONS         (formula + target)
#   external benchmarks             -> BENCHMARK_API_DB        (deliberately partial)
#
# The figures below are synthetic teaching data for one institution, AY 2025-26.
# Nothing in this notebook is ever computed from anything else: if a number
# appears in an answer, it came out of a tool call on these rows.

DEPARTMENT_REPORTS_CSV = """dept_code,dept_name,category,students_appeared,students_passed,co_po_attainment,sanctioned_faculty,publications,citations,grants_lakh,eligible_students,placed_students,total_package_lpa,interns_completed,budget_allocated_lakh,budget_spent_lakh,figure_status
CSE,Computer Science & Engineering,circuit,240,231,2.71,34,58,402,96.5,228,212,2247.2,205,420,393,final
ECE,Electronics & Communication,circuit,180,168,2.58,26,34,221,41.0,172,143,1058.2,149,310,276,final
ISE,Information Science & Engineering,circuit,120,116,2.66,18,21,140,22.5,114,103,947.6,96,240,221,final
EEE,Electrical & Electronics,circuit,120,109,2.41,17,16,88,18.0,116,86,524.6,84,210,181,final
MECH,Mechanical Engineering,core,180,158,2.33,24,19,95,27.5,174,111,599.4,102,300,247,final
CIVIL,Civil Engineering,core,120,101,2.19,16,11,47,12.0,118,62,285.2,61,190,149,final
CHEM,Chemical Engineering,core,60,54,2.28,11,14,76,15.5,58,33,168.3,34,140,118,final
BT,Biotechnology,core,60,55,2.35,10,12,63,9.5,57,29,142.1,30,130,102,final
"""

# Numeric columns, so the CSV reader returns typed values instead of strings.
NUMERIC_COLUMNS = {
    "students_appeared": int, "students_passed": int, "co_po_attainment": float,
    "sanctioned_faculty": int, "publications": int, "citations": int,
    "grants_lakh": float, "eligible_students": int, "placed_students": int,
    "total_package_lpa": float, "interns_completed": int,
    "budget_allocated_lakh": float, "budget_spent_lakh": float,
}

# ---- Approved KPI definitions -----------------------------------------------
# Each KPI declares HOW it is computed, not just what it is called. The compute
# tool is generic: it reads these definitions rather than hard-coding formulas,
# which is what stops an agent from quietly inventing its own arithmetic.
KPI_DEFINITIONS = {
    "KPI-A1": {"name": "Institution pass percentage", "domain": "Academic",
               "method": "ratio", "numerator": "students_passed",
               "denominator": "students_appeared", "scale": 100, "unit": "%",
               "formula": "sum(students_passed) / sum(students_appeared) x 100",
               "target": 92.0, "target_source": "Strategic Plan 2024-27",
               "direction": "higher_is_better", "source_system": "Academic ERP"},
    "KPI-A2": {"name": "CO/PO attainment index", "domain": "Academic",
               "method": "weighted_mean", "value_field": "co_po_attainment",
               "weight_field": "students_appeared", "unit": "index 0-3",
               "formula": "sum(co_po_attainment x students_appeared) / sum(students_appeared)",
               "target": 2.50, "target_source": "NAAC criterion 2",
               "direction": "higher_is_better", "source_system": "Academic ERP (OBE module)"},
    "KPI-R1": {"name": "Publications per faculty", "domain": "Research",
               "method": "ratio", "numerator": "publications",
               "denominator": "sanctioned_faculty", "scale": 1, "unit": "per faculty",
               "formula": "sum(publications) / sum(sanctioned_faculty)",
               "target": 1.50, "target_source": "NIRF Research & Professional Practice",
               "direction": "higher_is_better", "source_system": "Research repository"},
    "KPI-R2": {"name": "Citations per faculty", "domain": "Research",
               "method": "ratio", "numerator": "citations",
               "denominator": "sanctioned_faculty", "scale": 1, "unit": "per faculty",
               "formula": "sum(citations) / sum(sanctioned_faculty)",
               "target": 9.00, "target_source": "NIRF Research & Professional Practice",
               "direction": "higher_is_better", "source_system": "Research repository"},
    "KPI-R3": {"name": "Grants sanctioned", "domain": "Research",
               "method": "sum", "value_field": "grants_lakh", "unit": "INR lakh",
               "formula": "sum(grants_lakh)",
               "target": 300.0, "target_source": "Strategic Plan 2024-27",
               "direction": "higher_is_better", "source_system": "Research repository"},
    "KPI-P1": {"name": "Placement percentage", "domain": "Placement",
               "method": "ratio", "numerator": "placed_students",
               "denominator": "eligible_students", "scale": 100, "unit": "%",
               "formula": "sum(placed_students) / sum(eligible_students) x 100",
               "target": 80.0, "target_source": "Strategic Plan 2024-27",
               "direction": "higher_is_better", "source_system": "Placement Cell tracker"},
    "KPI-P2": {"name": "Average package", "domain": "Placement",
               "method": "ratio", "numerator": "total_package_lpa",
               "denominator": "placed_students", "scale": 1, "unit": "LPA",
               "formula": "sum(total_package_lpa) / sum(placed_students)",
               "target": 8.00, "target_source": "Strategic Plan 2024-27",
               "direction": "higher_is_better", "source_system": "Placement Cell tracker"},
    "KPI-P3": {"name": "Internship completion", "domain": "Placement",
               "method": "ratio", "numerator": "interns_completed",
               "denominator": "eligible_students", "scale": 100, "unit": "%",
               "formula": "sum(interns_completed) / sum(eligible_students) x 100",
               "target": 85.0, "target_source": "Strategic Plan 2024-27",
               "direction": "higher_is_better", "source_system": "Placement Cell tracker"},
    "KPI-F1": {"name": "Budget utilisation", "domain": "Finance",
               "method": "ratio", "numerator": "budget_spent_lakh",
               "denominator": "budget_allocated_lakh", "scale": 100, "unit": "%",
               "formula": "sum(budget_spent_lakh) / sum(budget_allocated_lakh) x 100",
               "target": 90.0, "target_source": "Strategic Plan 2024-27",
               "direction": "higher_is_better", "source_system": "Finance & Accounts ledger",
               "figure_status": "provisional",
               "caveat": "Q3-Q4 of 2025-26 are unreconciled; the Finance Office closes books in August."},
}

# ---- Mock external benchmark API --------------------------------------------
# Deliberately partial. Three KPIs have a published peer benchmark; the rest do
# not, and the assistant must say so rather than invent one.
BENCHMARK_API_DB = {
    "KPI-R1": {"peer_median": 1.82, "cohort": "NIRF Engineering rank band 51-100", "year": "2025"},
    "KPI-R2": {"peer_median": 11.40, "cohort": "NIRF Engineering rank band 51-100", "year": "2025"},
    "KPI-P1": {"peer_median": 78.60, "cohort": "NIRF Engineering rank band 51-100", "year": "2025"},
}

REPORTING_PERIOD = "academic year 2025-26"

print(f"Department reports : 8 departments, {len(NUMERIC_COLUMNS)} numeric columns, "
      f"{REPORTING_PERIOD}")
print(f"KPI definitions    : {len(KPI_DEFINITIONS)} approved KPIs "
      f"({sorted({k['domain'] for k in KPI_DEFINITIONS.values()})})")
print(f"External benchmarks: {len(BENCHMARK_API_DB)} of {len(KPI_DEFINITIONS)} KPIs have one "
      f"-> {sorted(set(KPI_DEFINITIONS) - set(BENCHMARK_API_DB))} do not")


Department reports : 8 departments, 13 numeric columns, academic year 2025-26
KPI definitions    : 9 approved KPIs (['Academic', 'Finance', 'Placement', 'Research'])
External benchmarks: 3 of 9 KPIs have one -> ['KPI-A1', 'KPI-A2', 'KPI-F1', 'KPI-P2', 'KPI-P3', 'KPI-R3'] do not


### Cell 4 · Tool contracts

Schemas first, implementations second. Writing the schemas as *data* rather than as docstrings is what lets them be reused three ways: input validation (Cell 5), Gemini function declarations (Cell 13), and the "tool definitions" deliverable (Cell 16).

`compute_kpi` deserves a second look: it is **generic**. It reads `method`, `numerator`, `denominator`, `scale` from the KPI registry and applies them. Adding a tenth KPI means adding a definition, not writing code — and no agent can quietly substitute its own formula.

In [4]:
# =============================================================================
# Cell 4  ·  Tool contracts — schemas first, then implementations
# =============================================================================
# Concept 2's tool discipline: a tool is a *contract*, not a function you happen
# to call. Every tool here declares
#
#   * a typed input schema (validated before the call — Cell 6),
#   * a documented return shape, and
#   * a single failure convention: an explicit {"error": ...} key.
#
# Nothing ever returns None-on-failure or an empty list that reads as "zero".
# The same schemas are reused as Gemini function declarations in Cell 13, which
# is exactly why they are written as data rather than as docstrings.

import csv, io, math

TOOL_SCHEMAS = {
    "read_department_reports": {
        "description": ("Read per-department metric rows from the department reports "
                        "export for one reporting period. Optionally restrict to named "
                        "departments or to a category (circuit / core)."),
        "parameters": {
            "type": "object",
            "properties": {
                "period": {"type": "string", "description": "e.g. 'academic year 2025-26'"},
                "departments": {"type": "array", "items": {"type": "string"},
                                "description": "dept_code list, e.g. ['CSE','ECE']"},
                "category": {"type": "string", "enum": ["circuit", "core"]},
            },
            "required": ["period"],
        },
        "returns": "{rows: [ {dept_code, ...typed metrics} ], row_count, source_system} | {error}",
    },
    "read_kpi_definitions": {
        "description": ("Read approved institution-wide KPI definitions: formula, unit, "
                        "target value and the authority the target comes from."),
        "parameters": {
            "type": "object",
            "properties": {"kpi_ids": {"type": "array", "items": {"type": "string"},
                                       "description": "e.g. ['KPI-P1']; omit for all"}},
            "required": [],
        },
        "returns": "{definitions: {kpi_id: {...}}, count} | {error}",
    },
    "read_external_benchmark": {
        "description": ("Call the external benchmark API for a KPI's published peer "
                        "median. Coverage is partial; many KPIs have no benchmark."),
        "parameters": {"type": "object",
                       "properties": {"kpi_id": {"type": "string"}},
                       "required": ["kpi_id"]},
        "returns": "{kpi_id, peer_median, cohort, year} | {error}",
    },
    "filter_rows": {
        "description": ("Filter previously-read department rows by an exact field value. "
                        "Errors if the filter matches nothing, so an empty subset can "
                        "never be silently averaged."),
        "parameters": {
            "type": "object",
            "properties": {
                "rows": {"type": "array", "items": {"type": "object"}},
                "field": {"type": "string"},
                "value": {"type": "string"},
            },
            "required": ["rows", "field", "value"],
        },
        "returns": "{rows, row_count, filter} | {error}",
    },
    "compute_kpi": {
        "description": ("Compute one approved KPI over a set of department rows, using "
                        "the KPI's registered formula. Returns the value plus per-"
                        "department drill-down evidence."),
        "parameters": {
            "type": "object",
            "properties": {
                "kpi_id": {"type": "string"},
                "rows": {"type": "array", "items": {"type": "object"}},
            },
            "required": ["kpi_id", "rows"],
        },
        "returns": ("{kpi_id, name, value, unit, formula, n_departments, inputs_used, "
                    "evidence: [...], source_system, figure_status} | {error}"),
    },
    "compare_to_target": {
        "description": ("Compare a computed KPI value against its approved target and "
                        "return the variance, the status band, and a rounding warning "
                        "if a rounded figure would hide a miss."),
        "parameters": {
            "type": "object",
            "properties": {"kpi_id": {"type": "string"}, "value": {"type": "number"}},
            "required": ["kpi_id", "value"],
        },
        "returns": ("{kpi_id, value, target, target_source, variance, pct_of_target, "
                    "status, rounding_warning} | {error}"),
    },
    "compute_variance": {
        "description": ("Compute the difference between a computed KPI value and any "
                        "reference figure (a peer benchmark, a prior year). Exists so "
                        "that no arithmetic is ever done in prose."),
        "parameters": {
            "type": "object",
            "properties": {"value": {"type": "number"}, "reference": {"type": "number"},
                           "label": {"type": "string"}},
            "required": ["value", "reference", "label"],
        },
        "returns": "{value, reference, label, variance, abs_variance, pct_of_reference} | {error}",
    },
    "draft_notification": {
        "description": ("DRAFT a notification for leadership. It is never sent: the "
                        "return value is a draft pending human approval. Every finding "
                        "must carry its source or the call is rejected."),
        "parameters": {
            "type": "object",
            "properties": {
                "audience": {"type": "string"},
                "subject": {"type": "string"},
                "findings": {"type": "array", "items": {"type": "object"},
                             "description": "each: {statement, source}"},
            },
            "required": ["audience", "subject", "findings"],
        },
        "returns": "{status: 'DRAFT - NOT SENT', audience, subject, body, requires_approval_from} | {error}",
    },
}


# ----------------------------------------------------------------------------
# Implementations. Each returns its documented shape, or {"error": ...}.
# ----------------------------------------------------------------------------
def _typed_rows(text: str) -> List[Dict[str, Any]]:
    rows = []
    for raw in csv.DictReader(io.StringIO(text)):
        row = dict(raw)
        for col, caster in NUMERIC_COLUMNS.items():
            if col in row:
                try:
                    row[col] = caster(row[col])
                except (TypeError, ValueError):
                    return [{"error": f"column '{col}' is not {caster.__name__} in row {raw}"}]
        rows.append(row)
    return rows


def read_department_reports(period: str, departments=None, category=None) -> Dict[str, Any]:
    if period != REPORTING_PERIOD:
        return {"error": (f"no department report export for '{period}'; "
                          f"the only loaded period is '{REPORTING_PERIOD}'")}
    rows = _typed_rows(DEPARTMENT_REPORTS_CSV)
    if rows and "error" in rows[0]:
        return {"error": rows[0]["error"]}
    if departments:
        known = {r["dept_code"] for r in rows}
        unknown = [d for d in departments if d not in known]
        if unknown:
            return {"error": f"unknown department code(s): {unknown}; known: {sorted(known)}"}
        rows = [r for r in rows if r["dept_code"] in departments]
    if category:
        rows = [r for r in rows if r["category"] == category]
        if not rows:
            return {"error": f"no departments in category '{category}'"}
    if not rows:
        return {"error": "empty department report export"}
    return {"rows": rows, "row_count": len(rows), "period": period,
            "source_system": "Department reports export (Registrar-authorised read)"}


def read_kpi_definitions(kpi_ids=None) -> Dict[str, Any]:
    if kpi_ids:
        unknown = [k for k in kpi_ids if k not in KPI_DEFINITIONS]
        if unknown:
            return {"error": f"unknown KPI id(s): {unknown}; known: {sorted(KPI_DEFINITIONS)}"}
        defs = {k: KPI_DEFINITIONS[k] for k in kpi_ids}
    else:
        defs = dict(KPI_DEFINITIONS)
    return {"definitions": defs, "count": len(defs),
            "source_system": "Institutional KPI register (approved definitions)"}


def read_external_benchmark(kpi_id: str) -> Dict[str, Any]:
    if kpi_id not in KPI_DEFINITIONS:
        return {"error": f"unknown KPI id '{kpi_id}'"}
    record = BENCHMARK_API_DB.get(kpi_id)
    if record is None:
        return {"error": (f"no published external benchmark for {kpi_id} "
                          f"({KPI_DEFINITIONS[kpi_id]['name']}) in the benchmark API")}
    return {"kpi_id": kpi_id, **record, "source_system": "External benchmark API (NIRF cohort)"}


def filter_rows(rows: List[Dict[str, Any]], field: str, value: str) -> Dict[str, Any]:
    if not rows:
        return {"error": "filter_rows received an empty row set"}
    if field not in rows[0]:
        return {"error": f"unknown field '{field}'; available: {sorted(rows[0])}"}
    subset = [r for r in rows if str(r[field]) == str(value)]
    if not subset:
        return {"error": (f"filter {field}={value!r} matched 0 of {len(rows)} rows — "
                          f"refusing to return an empty subset")}
    return {"rows": subset, "row_count": len(subset),
            "filter": f"{field}=={value}", "filtered_from": len(rows)}


def compute_kpi(kpi_id: str, rows: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Generic: the arithmetic comes from the KPI definition, not from this code."""
    if kpi_id not in KPI_DEFINITIONS:
        return {"error": f"unknown KPI id '{kpi_id}'"}
    if not rows:
        return {"error": f"cannot compute {kpi_id} over an empty row set"}
    d = KPI_DEFINITIONS[kpi_id]
    method, evidence = d["method"], []

    try:
        if method == "ratio":
            num_f, den_f, scale = d["numerator"], d["denominator"], d["scale"]
            num = sum(r[num_f] for r in rows)
            den = sum(r[den_f] for r in rows)
            if den == 0:
                return {"error": f"{kpi_id}: denominator sum({den_f}) is zero"}
            value = num / den * scale
            inputs = {f"sum({num_f})": round(num, 4), f"sum({den_f})": round(den, 4)}
            for r in rows:
                dv = (r[num_f] / r[den_f] * scale) if r[den_f] else None
                evidence.append({"dept_code": r["dept_code"], "dept_name": r["dept_name"],
                                 num_f: r[num_f], den_f: r[den_f],
                                 "dept_value": round(dv, 2) if dv is not None else None,
                                 "share_of_denominator_pct": round(r[den_f] / den * 100, 2)})
        elif method == "weighted_mean":
            vf, wf = d["value_field"], d["weight_field"]
            wsum = sum(r[wf] for r in rows)
            if wsum == 0:
                return {"error": f"{kpi_id}: total weight sum({wf}) is zero"}
            value = sum(r[vf] * r[wf] for r in rows) / wsum
            inputs = {f"sum({vf} x {wf})": round(sum(r[vf] * r[wf] for r in rows), 4),
                      f"sum({wf})": wsum}
            for r in rows:
                evidence.append({"dept_code": r["dept_code"], "dept_name": r["dept_name"],
                                 vf: r[vf], wf: r[wf],
                                 "dept_value": r[vf],
                                 "weight_pct": round(r[wf] / wsum * 100, 2)})
        elif method == "sum":
            vf = d["value_field"]
            value = sum(r[vf] for r in rows)
            inputs = {f"sum({vf})": round(value, 4)}
            for r in rows:
                evidence.append({"dept_code": r["dept_code"], "dept_name": r["dept_name"],
                                 vf: r[vf], "dept_value": r[vf],
                                 "share_of_total_pct": round(r[vf] / value * 100, 2) if value else None})
        else:
            return {"error": f"unsupported KPI method '{method}' for {kpi_id}"}
    except KeyError as missing:
        return {"error": f"{kpi_id}: department rows are missing column {missing}"}

    statuses = {r.get("figure_status", "final") for r in rows}
    figure_status = d.get("figure_status") or ("final" if statuses == {"final"} else "mixed")
    out = {"kpi_id": kpi_id, "name": d["name"], "domain": d["domain"],
           "value": round(value, 4), "unit": d["unit"], "formula": d["formula"],
           "method": method, "n_departments": len(rows),
           "departments": [r["dept_code"] for r in rows],
           "inputs_used": inputs, "evidence": evidence,
           "source_system": d["source_system"], "figure_status": figure_status}
    if d.get("caveat"):
        out["caveat"] = d["caveat"]
    return out


def compare_to_target(kpi_id: str, value: float) -> Dict[str, Any]:
    if kpi_id not in KPI_DEFINITIONS:
        return {"error": f"unknown KPI id '{kpi_id}'"}
    d = KPI_DEFINITIONS[kpi_id]
    target = d["target"]
    variance = value - target
    pct_of_target = value / target * 100 if target else None
    gap_pct = abs(variance) / target * 100 if target else 0
    if variance >= 0:
        status = "on target"
    elif gap_pct <= 2.0:
        status = "at risk"
    else:
        status = "below target"

    # Guardrail: "never mask a missed target in a rounded figure."
    rounding_warning = None
    if variance < 0:
        for places in (0, 1):
            if round(value, places) >= target:
                rounding_warning = (
                    f"{value:.4g} rounds to {round(value, places)} at {places} decimal place(s), "
                    f"which would read as meeting the target of {target}. Report the "
                    f"unrounded figure.")
                break
    return {"kpi_id": kpi_id, "name": d["name"], "value": round(value, 4),
            "target": target, "target_source": d["target_source"],
            "unit": d["unit"], "variance": round(variance, 4),
            "pct_of_target": round(pct_of_target, 2) if pct_of_target else None,
            "status": status, "rounding_warning": rounding_warning}


def compute_variance(value: float, reference: float, label: str) -> Dict[str, Any]:
    """Any subtraction that ends up in an answer happens here, not in an f-string."""
    if reference == 0:
        return {"error": f"cannot compute variance against a zero reference ({label})"}
    variance = value - reference
    return {"value": round(value, 4), "reference": round(reference, 4), "label": label,
            "variance": round(variance, 4), "abs_variance": round(abs(variance), 4),
            "pct_of_reference": round(value / reference * 100, 2)}


def draft_notification(audience: str, subject: str,
                       findings: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Drafts only. Nothing is sent — Chapter 15 allows zero consequential
    actions without the executive's approval."""
    if not findings:
        return {"error": "refusing to draft a notification with no findings"}
    unsourced = [f for f in findings if not f.get("source")]
    if unsourced:
        return {"error": (f"{len(unsourced)} finding(s) carry no source; every figure in a "
                          "leadership notification must cite where it came from")}
    body = "\n".join(f"  {i}. {f['statement']}\n     source: {f['source']}"
                     for i, f in enumerate(findings, 1))
    return {"status": "DRAFT - NOT SENT", "audience": audience, "subject": subject,
            "body": body, "findings_cited": len(findings),
            "requires_approval_from": "Principal / Registrar (Chapter 15 human gate)",
            "note": "This tool cannot send. Circulation happens only after human approval."}


TOOLS = {
    "read_department_reports": read_department_reports,
    "read_kpi_definitions": read_kpi_definitions,
    "read_external_benchmark": read_external_benchmark,
    "filter_rows": filter_rows,
    "compute_kpi": compute_kpi,
    "compare_to_target": compare_to_target,
    "compute_variance": compute_variance,
    "draft_notification": draft_notification,
}

print(f"{len(TOOLS)} tools defined, each with a typed input schema and an explicit "
      f"error contract:\n")
for name, spec in TOOL_SCHEMAS.items():
    req = spec["parameters"].get("required", [])
    opt = [p for p in spec["parameters"]["properties"] if p not in req]
    print(f"  {name}")
    print(f"      required: {req or '[]'}   optional: {opt or '[]'}")
    print(f"      returns : {spec['returns']}")


8 tools defined, each with a typed input schema and an explicit error contract:

  read_department_reports
      required: ['period']   optional: ['departments', 'category']
      returns : {rows: [ {dept_code, ...typed metrics} ], row_count, source_system} | {error}
  read_kpi_definitions
      required: []   optional: ['kpi_ids']
      returns : {definitions: {kpi_id: {...}}, count} | {error}
  read_external_benchmark
      required: ['kpi_id']   optional: []
      returns : {kpi_id, peer_median, cohort, year} | {error}
  filter_rows
      required: ['rows', 'field', 'value']   optional: []
      returns : {rows, row_count, filter} | {error}
  compute_kpi
      required: ['kpi_id', 'rows']   optional: []
      returns : {kpi_id, name, value, unit, formula, n_departments, inputs_used, evidence: [...], source_system, figure_status} | {error}
  compare_to_target
      required: ['kpi_id', 'value']   optional: []
      returns : {kpi_id, value, target, target_source, variance, pct_of_tar

### Cell 5 · The choke point, and the connector over it

`ToolRouter.call()` is the only path to a tool:

1. **validate** the arguments against the schema — *before* the call,
2. **execute**,
3. **log** tool, reason, input and output,
4. **raise** on `{"error": ...}`; `try_call` is the explicit opt-in for the graceful-failure path.

Step 1 is the instructor check *"are inputs structured?"*, answered by construction: four malformed calls are attempted at the end of the cell and all four are rejected before reaching a tool.

`Connector` sits on top and names *resources* rather than functions — the Lab 5 seed. Swapping a source later will not require touching a single agent.

In [5]:
# =============================================================================
# Cell 5  ·  The single choke point — ToolRouter — and the connector over it
# =============================================================================
# Every tool call in this notebook goes through ToolRouter.call(). Nothing calls
# a tool function directly. The router does four things, in this order:
#
#   1. VALIDATES THE INPUT against the tool's schema *before* the call.
#      This is the instructor check "are inputs structured?" — a wrong type or a
#      missing required argument is rejected here, not discovered halfway
#      through the tool body.
#   2. EXECUTES the tool.
#   3. LOGS tool name, the reason the planner chose it, the exact structured
#      input and the output — the execution log the deliverable asks for.
#   4. RAISES on {"error": ...} so a bad result can never propagate silently.
#      `try_call` is the explicit opt-in for the graceful-failure path.
#
# The Connector on top of it is the Lab 5 seed: agents name a *resource*, not a
# function, so swapping a source later needs no change in any agent.

class ToolContractError(Exception):
    """Raised before a call when the arguments do not satisfy the tool schema."""


class ToolError(Exception):
    """Raised after a call when the tool returned an explicit error."""


_JSON_TYPES = {"string": str, "number": (int, float), "integer": int,
               "boolean": bool, "array": list, "object": dict}


def _summarise(obj, _depth=0):
    """Shrink big row lists so the trace stays readable without hiding structure."""
    if isinstance(obj, list):
        if len(obj) > 3 and all(isinstance(x, dict) for x in obj):
            return f"[{len(obj)} rows: {', '.join(str(x.get('dept_code', '?')) for x in obj)}]"
        return [_summarise(x, _depth + 1) for x in obj]
    if isinstance(obj, dict):
        return {k: _summarise(v, _depth + 1) for k, v in obj.items()}
    if isinstance(obj, str) and len(obj) > 160:
        return obj[:157] + "..."
    return obj


class ToolRouter:
    def __init__(self):
        self.trace: List[Dict[str, Any]] = []
        # Un-summarised outputs, kept for the grounding check in Cell 14: the
        # printed trace collapses long row lists, and a grounding check must
        # see every figure a tool actually produced.
        self.raw_outputs: List[Any] = []

    # ---- 1. input validation -------------------------------------------
    def validate_input(self, tool: str, kwargs: Dict[str, Any]) -> None:
        if tool not in TOOL_SCHEMAS:
            raise ToolContractError(f"no such tool: '{tool}'")
        schema = TOOL_SCHEMAS[tool]["parameters"]
        props, required = schema["properties"], schema.get("required", [])

        for name in required:
            if name not in kwargs:
                raise ToolContractError(f"{tool}: missing required argument '{name}'")
        for name, val in kwargs.items():
            if name not in props:
                raise ToolContractError(
                    f"{tool}: unknown argument '{name}'; accepts {sorted(props)}")
            spec = props[name]
            expected = _JSON_TYPES.get(spec["type"])
            if expected and not isinstance(val, expected):
                raise ToolContractError(
                    f"{tool}.{name}: expected {spec['type']}, got "
                    f"{type(val).__name__} ({val!r})")
            if spec.get("enum") and val not in spec["enum"]:
                raise ToolContractError(
                    f"{tool}.{name}: {val!r} not in {spec['enum']}")
            if spec["type"] == "array" and spec.get("items", {}).get("type"):
                item_type = _JSON_TYPES.get(spec["items"]["type"])
                bad = [v for v in val if item_type and not isinstance(v, item_type)]
                if bad:
                    raise ToolContractError(
                        f"{tool}.{name}: array items must be {spec['items']['type']}; "
                        f"offending: {bad[:2]}")

    # ---- 2-4. execute, log, check --------------------------------------
    def call(self, tool: str, reason: str, **kwargs) -> Dict[str, Any]:
        self.validate_input(tool, kwargs)
        result = TOOLS[tool](**kwargs)
        entry = {"step": len(self.trace) + 1, "tool": tool, "reason": reason,
                 "input": _summarise(kwargs), "output": _summarise(result),
                 "ok": "error" not in result}
        self.trace.append(entry)
        self.raw_outputs.append(result)
        if "error" in result:
            raise ToolError(f"{tool} failed: {result['error']}")
        return result

    def try_call(self, tool: str, reason: str, **kwargs) -> Dict[str, Any]:
        """Same contract, but hands the error back instead of raising.

        Used where a missing record is a legitimate answer ("we don't have
        that") rather than a failure of the run.
        """
        try:
            return self.call(tool, reason, **kwargs)
        except ToolError:
            return dict(TOOLS[tool](**kwargs))

    def calls_to(self, tool: str) -> List[Dict[str, Any]]:
        return [e for e in self.trace if e["tool"] == tool]

    def sequence(self) -> List[str]:
        return [e["tool"] for e in self.trace]


class Connector:
    """One governed surface over every source. No agent touches a source directly."""

    RESOURCES = {
        "department reports":              "read_department_reports",
        "institutional KPIs & dashboards": "read_kpi_definitions",
        "external benchmarks":             "read_external_benchmark",
    }

    def __init__(self, router: ToolRouter):
        self.router = router

    def read(self, resource: str, reason: str, required: bool = True, **params):
        if resource not in self.RESOURCES:
            raise ToolContractError(
                f"connector exposes no resource '{resource}'; "
                f"available: {sorted(self.RESOURCES)}")
        tool = self.RESOURCES[resource]
        caller = self.router.call if required else self.router.try_call
        return caller(tool, reason, **params)


# ---- The contract is real: prove the guard rejects bad input ----------------
_probe_router = ToolRouter()
for bad_call, why in [
    (dict(tool="read_department_reports", reason="probe"), "missing required 'period'"),
    (dict(tool="compare_to_target", reason="probe", kpi_id="KPI-P1", value="75.1"),
     "'value' is a string, schema says number"),
    (dict(tool="read_department_reports", reason="probe",
          period=REPORTING_PERIOD, category="circuits"), "'circuits' is not in the enum"),
    (dict(tool="read_department_reports", reason="probe",
          period=REPORTING_PERIOD, year=2026), "unknown argument 'year'"),
]:
    tool = bad_call.pop("tool"); reason = bad_call.pop("reason")
    try:
        _probe_router.call(tool, reason, **bad_call)
        print(f"  [!!] NOT REJECTED — {why}")
    except ToolContractError as exc:
        print(f"  [rejected before the call] {exc}")
print(f"\n  trace length after 4 rejected calls: {len(_probe_router.trace)} "
      f"(bad input never reaches a tool, so it never reaches the log)")


  [rejected before the call] read_department_reports: missing required argument 'period'
  [rejected before the call] compare_to_target.value: expected number, got str ('75.1')
  [rejected before the call] read_department_reports.category: 'circuits' not in ['circuit', 'core']
  [rejected before the call] read_department_reports: unknown argument 'year'; accepts ['category', 'departments', 'period']

  trace length after 4 rejected calls: 0 (bad input never reaches a tool, so it never reaches the log)


### Cell 6 · The validation gate — `assert v.valid`

Chapter 15's Lab 2 pseudocode ends on that line. A computed KPI is only usable if it can prove where it came from: the formula used, the source system, the raw inputs, per-department evidence covering exactly the departments computed over, and a caveat on anything not final.

The gate is not advisory. `kpi_chain()` asserts on it, so a KPI that cannot show its evidence never reaches an answer.

In [6]:
# =============================================================================
# Cell 6  ·  The validation gate —  assert v.valid
# =============================================================================
# Chapter 15's Lab 2 pseudocode ends on `assert v.valid`. This is that gate.
# A computed KPI is only usable if it can prove where it came from:

class Validated:
    """Result of the acceptance gate. Truthy only when every check passed."""

    def __init__(self, checks: List[tuple], subject: str):
        self.checks = checks
        self.subject = subject
        self.failing = [name for name, ok, _ in checks if not ok]
        self.valid = not self.failing

    def __repr__(self):
        return (f"<Validated {self.subject}: "
                f"{'VALID' if self.valid else 'INVALID ' + str(self.failing)}>")

    def report(self, indent="    ") -> str:
        return "\n".join(f"{indent}[{'ok' if ok else 'FAIL'}] {name}"
                         + (f"  ({detail})" if detail else "")
                         for name, ok, detail in self.checks)


def validate_kpi_result(computed: Dict[str, Any],
                        compared: Optional[Dict[str, Any]] = None,
                        expected_departments: Optional[int] = None) -> Validated:
    ev = computed.get("evidence", [])
    checks = [
        ("value is a finite number",
         isinstance(computed.get("value"), (int, float)) and math.isfinite(computed["value"]),
         f"value={computed.get('value')}"),
        ("the formula used is recorded",
         bool(computed.get("formula")), computed.get("formula", "")),
        ("the figure cites its source system",
         bool(computed.get("source_system")), computed.get("source_system", "")),
        ("raw inputs are recorded, not just the result",
         bool(computed.get("inputs_used")), str(computed.get("inputs_used", ""))),
        ("every department in the evidence is identified",
         bool(ev) and all(e.get("dept_code") for e in ev), f"{len(ev)} evidence rows"),
        ("evidence covers exactly the departments computed over",
         len(ev) == computed.get("n_departments"),
         f"{len(ev)} evidence rows vs n_departments={computed.get('n_departments')}"),
        ("provisional figures carry a caveat",
         computed.get("figure_status") == "final" or bool(computed.get("caveat")),
         f"figure_status={computed.get('figure_status')}"),
    ]
    if expected_departments is not None:
        checks.append(("department count matches the requested scope",
                       computed.get("n_departments") == expected_departments,
                       f"{computed.get('n_departments')} of {expected_departments}"))
    if compared is not None:
        checks += [
            ("the target names the authority that set it",
             bool(compared.get("target_source")), compared.get("target_source", "")),
            ("a rounding that would hide a miss is flagged",
             True, compared.get("rounding_warning") or "no rounding risk"),
        ]
    return Validated(checks, computed.get("kpi_id", "?"))


print("Validation gate ready.  Usage:  v = validate_kpi_result(...);  assert v.valid")


Validation gate ready.  Usage:  v = validate_kpi_result(...);  assert v.valid


### Cell 7 · The assistant, and the grounding gate

Two rules govern every answer:

1. **No number is written by the model.** Values come out of `compute_kpi` and `compare_to_target`.
2. **That rule is enforced, not trusted.** `ground_check()` extracts every numeric token from a composed answer and rejects it if the token does not appear in the tool outputs it was built from. A rejected answer falls back to the deterministic composition and the rejection is logged.

The same gate is applied to the deterministic text too — a rule that only polices the model is not a rule.

In [7]:
# =============================================================================
# Cell 7  ·  The assistant — plan a tool chain, run it, ground the answer
# =============================================================================
# Two rules govern every answer produced below:
#
#   1. NO NUMBER IS EVER WRITTEN BY THE MODEL. Values come out of compute_kpi
#      and compare_to_target. The wording may be model-generated; the figures
#      may not be.
#   2. THAT RULE IS ENFORCED, NOT TRUSTED. `ground_check()` extracts every
#      numeric token from a composed answer and rejects the answer if a token
#      does not appear in the tool outputs it was built from. A rejected answer
#      falls back to the deterministic composition, and the rejection is logged.

def allowed_numbers(objs: List[Any]) -> set:
    """Every number a tool actually produced, at the precisions we might print."""
    found, out = [], set()

    def walk(o):
        if isinstance(o, dict):
            [walk(v) for v in o.values()]
        elif isinstance(o, list):
            [walk(v) for v in o]
        elif isinstance(o, bool):
            pass
        elif isinstance(o, (int, float)):
            found.append(float(o))
        elif isinstance(o, str):
            out.update(re.findall(r"\d+(?:\.\d+)?", o))     # numbers inside formulas/text

    walk(objs)
    # Both signs: a tool records variance as -4.88, while the answer text says
    # "4.88 percentage points below". Same figure, so both spellings are allowed.
    found = [x for n in found for x in (n, abs(n))]
    for n in found:
        for places in (0, 1, 2, 3, 4):
            out.add(f"{n:.{places}f}".rstrip("0").rstrip(".") if places else f"{n:.0f}")
            out.add(f"{n:.{places}f}")
        out.add(str(n))
        if float(n).is_integer():
            out.add(str(int(n)))
    return out


PERIOD_TOKENS = {"2025", "26", "2024", "27", "2023", "2026", "15", "2"}


def ground_check(answer: str, tool_outputs: List[Any]) -> tuple:
    allowed = allowed_numbers(tool_outputs) | PERIOD_TOKENS
    ungrounded = [tok for tok in re.findall(r"\d+(?:\.\d+)?", answer) if tok not in allowed]
    return (not ungrounded), ungrounded


COMPOSE_SCHEMA = {"type": "object",
                  "properties": {"answer": {"type": "string"}},
                  "required": ["answer"]}
COMPOSE_SYSTEM = """You write one short, factual paragraph for the Principal / Registrar of an
educational institution. You may ONLY use the numbers given to you. You may not compute,
round, estimate, infer or add any figure that is not in the input. If a figure is marked
provisional you must say so."""


@llm.offline.register("compose")
def _offline_compose(payload):
    return {"answer": payload["deterministic"]}


class ExecutiveKPIAssistant:
    """Lab 2 scope: Coordinator + tool layer. No specialist agents yet."""

    def __init__(self):
        self.router = ToolRouter()
        self.connector = Connector(self.router)
        self.answers: List[Dict[str, Any]] = []
        self.validations: List[Validated] = []
        self.rejected_compositions: List[Dict[str, Any]] = []

    # ---- the reusable KPI chain: read -> define -> compute -> compare ----
    def kpi_chain(self, kpi_id: str, why: str, rows=None,
                  scope_label: str = "all 8 departments") -> Dict[str, Any]:
        if rows is None:
            rows = self.connector.read(
                "department reports",
                reason=f"{why}: the per-department metrics live in the reports export",
                period=REPORTING_PERIOD)["rows"]
        definition = self.connector.read(
            "institutional KPIs & dashboards",
            reason=f"{why}: the approved formula and target must come from the register, "
                   "not from the prompt",
            kpi_ids=[kpi_id])["definitions"][kpi_id]
        computed = self.router.call(
            "compute_kpi",
            reason=f"{why}: apply the registered formula over {scope_label}",
            kpi_id=kpi_id, rows=rows)
        compared = self.router.call(
            "compare_to_target",
            reason=f"{why}: variance against the approved target is what leadership acts on",
            kpi_id=kpi_id, value=computed["value"])

        v = validate_kpi_result(computed, compared, expected_departments=len(rows))
        self.validations.append(v)
        assert v.valid, f"validation gate failed for {kpi_id}: {v.failing}"   # hard gate

        return {"definition": definition, "computed": computed,
                "compared": compared, "validation": v, "rows": rows}

    # ---- answer composition, with the grounding + disclosure gates -------
    def compose(self, deterministic: str, tool_outputs: List[Any], topic: str,
                must_include: tuple = ()) -> str:
        """Optionally let the model reword — but only if it keeps every figure
        AND every mandatory disclosure.

        `must_include` carries the Chapter 15 disclosure requirements for this
        answer: the source system behind each figure, the PROVISIONAL label on
        an unreconciled figure, the ROUNDING WARNING where a rounded number
        would read as on-target. These are compliance text, not house style, so
        a reworded answer that drops one is rejected outright.
        """
        ok, bad = ground_check(deterministic, tool_outputs)
        if not ok:                      # our own text must pass the gate too
            self.rejected_compositions.append(
                {"topic": topic, "source": "deterministic", "ungrounded": bad})
            return deterministic
        missing_here = [p for p in must_include if p not in deterministic]
        if missing_here:                # a bug in the handler, not in the model
            self.rejected_compositions.append(
                {"topic": topic, "source": "deterministic",
                 "missing_disclosures": missing_here})
            return deterministic
        try:
            mandatory = ("\n\nMANDATORY — reproduce each of these strings VERBATIM, "
                         "character for character, somewhere in your answer:\n"
                         + "\n".join(f"  {p!r}" for p in must_include)
                         if must_include else "")
            polished = llm.json(
                task="compose", system=COMPOSE_SYSTEM,
                prompt=(f"Rewrite this for the Principal, keeping every figure exactly as "
                        f"written and adding none:\n\n{deterministic}\n\n"
                        f"Permitted figures, and nothing else:\n"
                        f"{json.dumps(_summarise(tool_outputs), indent=2)}{mandatory}"),
                schema=COMPOSE_SCHEMA,
                payload={"deterministic": deterministic}).get("answer") or deterministic
        except Exception as exc:          # a model outage must not lose the answer
            self.rejected_compositions.append(
                {"topic": topic, "source": llm.mode, "ungrounded": [],
                 "error": f"{type(exc).__name__}: {exc}"})
            return deterministic
        ok, bad = ground_check(polished, tool_outputs)
        if not ok:
            self.rejected_compositions.append(
                {"topic": topic, "source": llm.mode, "ungrounded": bad,
                 "rejected_text": polished})
            return deterministic        # fall back rather than publish a stray number
        dropped = [p for p in must_include if p not in polished]
        if dropped:
            self.rejected_compositions.append(
                {"topic": topic, "source": llm.mode, "dropped_disclosures": dropped,
                 "rejected_text": polished})
            return deterministic        # a dropped disclosure is a failed rewrite
        return polished

    def record(self, question: str, answer: str, route: str, steps_before: int):
        self.answers.append({"question": question, "answer": answer, "route": route,
                             "tools_used": self.router.sequence()[steps_before:]})
        return answer


### Cell 8 · The planner

*"Use available tools only when needed. Explain why each tool was called."*

Route selection is a recorded decision, and every call inside the selected chain carries its own `reason` string into the execution log. The last route is a deliberate dead end: a question with no tool path gets a refusal, not an answer from memory.

In [8]:
# =============================================================================
# Cell 8  ·  The planner — decide whether a tool is needed, and which
# =============================================================================
# "Use available tools only when needed. Explain why each tool was called."
# Routing is a decision recorded in the trace: the assistant states which chain
# it selected and why, then every call inside that chain carries its own reason.

def _fmt(value, unit):
    if unit == "%":
        return f"{value:.2f}%"
    if unit == "LPA":
        return f"{value:.2f} LPA"
    if unit == "INR lakh":
        return f"INR {value:.2f} lakh"
    if unit.startswith("index"):
        return f"{value:.3f} (scale 0-3)"
    return f"{value:.3f} {unit}"


class CampusExecutiveAssistant(ExecutiveKPIAssistant):

    # Order matters: the more specific pattern must be tested first, or a
    # drill-down question would be absorbed by the general placement route.
    ROUTES = [
        (r"(which|what) departments?|drill[- ]?down|breakdown by department",
         "placement_drilldown"),
        (r"(circuit|core)[- ]?(branch|stream)", "branch_placement"),
        (r"placement (percentage|rate|%)?.*(target|strategic|doing|standing)|"
         r"(target|strategic).*placement", "placement_vs_target"),
        (r"citation.*(peer|benchmark|compare|other institution|nirf)|"
         r"(peer|benchmark).*citation", "research_vs_peers"),
        (r"internship.*(peer|benchmark|compare|nirf)", "internship_vs_peers"),
        (r"budget|utilisation|utilization|spend", "budget_utilisation"),
        (r"pass (percentage|rate|%)|academic performance", "pass_percentage"),
    ]

    def route(self, question: str) -> str:
        q = re.sub(r"\s+", " ", question.lower())
        for pattern, name in self.ROUTES:
            if re.search(pattern, q):
                return name
        return "unrouted"

    # ---------------------------------------------------------------- Q1
    def placement_vs_target(self, q):
        r = self.kpi_chain("KPI-P1", "leadership asked how placement stands against target")
        c, cmp_ = r["computed"], r["compared"]
        text = (f"Institution-wide placement for {REPORTING_PERIOD} is "
                f"{_fmt(c['value'], c['unit'])}, computed as {c['formula']} over "
                f"{c['n_departments']} departments "
                f"({', '.join(f'{k}={v}' for k, v in c['inputs_used'].items())}). "
                f"The {cmp_['target_source']} target is {cmp_['target']:.2f}%, so we are "
                f"{abs(cmp_['variance']):.2f} percentage points below it — status: "
                f"{cmp_['status']}. Source: {c['source_system']}.")
        return self.compose(text, [c, cmp_], "placement_vs_target",
                            must_include=(c["source_system"],))

    # ---------------------------------------------------------------- Q2
    def placement_drilldown(self, q):
        r = self.kpi_chain("KPI-P1", "leadership asked which departments pull placement down")
        c, cmp_ = r["computed"], r["compared"]
        ranked = sorted(c["evidence"], key=lambda e: e["dept_value"])
        worst = ranked[:3]
        lines = "; ".join(
            f"{e['dept_code']} at {e['dept_value']:.2f}% "
            f"({e['placed_students']} of {e['eligible_students']} eligible, "
            f"{e['share_of_denominator_pct']:.2f}% of the eligible pool)"
            for e in worst)
        text = (f"Placement is {_fmt(c['value'], c['unit'])} against a target of "
                f"{cmp_['target']:.2f}%. The departments furthest below the institution "
                f"figure are: {lines}. Each figure traces to {c['source_system']} via the "
                f"department reports export, so leadership can follow any number back to "
                f"its department row.")
        return self.compose(text, [c, cmp_], "placement_drilldown",
                            must_include=(c["source_system"],))

    # ---------------------------------------------------------------- Q3
    def research_vs_peers(self, q):
        r = self.kpi_chain("KPI-R2", "leadership asked how research citations compare to peers")
        c, cmp_ = r["computed"], r["compared"]
        bench = self.connector.read(
            "external benchmarks",
            reason="a peer comparison needs the published benchmark, which is not in our own data",
            required=False, kpi_id="KPI-R2")
        if "error" in bench:
            return self.compose(
                f"Citations per faculty is {_fmt(c['value'], c['unit'])}, but "
                f"{bench['error']}.", [c, cmp_], "research_vs_peers",
                must_include=(bench["error"],))
        gap = self.router.call(
            "compute_variance",
            reason="the gap to the peer median is arithmetic, so it goes through a tool",
            value=c["value"], reference=bench["peer_median"],
            label="citations per faculty vs NIRF cohort median")
        text = (f"Citations per faculty is {c['value']:.2f} "
                f"({c['formula']}: {', '.join(f'{k}={v}' for k, v in c['inputs_used'].items())}). "
                f"The {bench['cohort']} peer median for {bench['year']} is "
                f"{bench['peer_median']:.2f}, so we sit {gap['abs_variance']:.2f} below the "
                f"cohort, and {abs(cmp_['variance']):.2f} below our own "
                f"{cmp_['target_source']} target of {cmp_['target']:.2f}. Sources: "
                f"{c['source_system']}; {bench['source_system']}.")
        return self.compose(text, [c, cmp_, bench, gap], "research_vs_peers",
                            must_include=(c["source_system"], bench["source_system"]))

    # ---------------------------------------------------------------- Q4
    def internship_vs_peers(self, q):
        r = self.kpi_chain("KPI-P3", "leadership asked about internship completion vs peers")
        c, cmp_ = r["computed"], r["compared"]
        bench = self.connector.read(
            "external benchmarks",
            reason="checking whether a peer benchmark exists for internship completion",
            required=False, kpi_id="KPI-P3")
        if "error" in bench:
            text = (f"Internship completion is {_fmt(c['value'], c['unit'])} against our "
                    f"{cmp_['target_source']} target of {cmp_['target']:.2f}% "
                    f"({cmp_['status']}, {abs(cmp_['variance']):.2f} points short). "
                    f"I cannot give you a peer comparison: {bench['error']}. "
                    f"I am not going to estimate one. Source: {c['source_system']}.")
            # The honest-failure statement and the API's own error text are
            # disclosures, not phrasing: a rewrite may not drop either.
            return self.compose(text, [c, cmp_], "internship_vs_peers",
                                must_include=(c["source_system"], bench["error"],
                                              "I am not going to estimate one."))
        return self.compose(f"Internship completion is {_fmt(c['value'], c['unit'])}; "
                            f"peer median {bench['peer_median']}. "
                            f"Source: {c['source_system']}.",
                            [c, cmp_, bench], "internship_vs_peers",
                            must_include=(c["source_system"],))

    # ---------------------------------------------------------------- Q5
    def budget_utilisation(self, q):
        r = self.kpi_chain("KPI-F1", "leadership asked about budget utilisation")
        c, cmp_ = r["computed"], r["compared"]
        text = (f"PROVISIONAL FIGURE. Budget utilisation is "
                f"{_fmt(c['value'], c['unit'])} ({c['formula']}: "
                f"{', '.join(f'{k}={v}' for k, v in c['inputs_used'].items())}), against a "
                f"{cmp_['target_source']} target of {cmp_['target']:.2f}% — {cmp_['status']}. "
                f"This figure is not final: {c['caveat']} It must not be presented to the "
                f"Governing Council as an actual. Source: {c['source_system']}.")
        # "PROVISIONAL FIGURE" and the reconciliation caveat are the Finance
        # Agent guardrail in text form. A rewrite that loses them is rejected.
        return self.compose(text, [c, cmp_], "budget_utilisation",
                            must_include=("PROVISIONAL FIGURE", c["caveat"],
                                          c["source_system"]))

    # ---------------------------------------------------------------- Q6
    def pass_percentage(self, q):
        r = self.kpi_chain("KPI-A1", "leadership asked whether the pass-percentage target is met")
        c, cmp_ = r["computed"], r["compared"]
        warn = (" ROUNDING WARNING: " + cmp_["rounding_warning"]) if cmp_["rounding_warning"] else ""
        text = (f"Institution pass percentage is {c['value']:.2f}% "
                f"({', '.join(f'{k}={v}' for k, v in c['inputs_used'].items())}) against the "
                f"{cmp_['target_source']} target of {cmp_['target']:.2f}%. That is a miss of "
                f"{abs(cmp_['variance']):.2f} points — status: {cmp_['status']}.{warn} "
                f"Source: {c['source_system']}.")
        required = [c["source_system"]]
        if cmp_["rounding_warning"]:
            required.append("ROUNDING WARNING")
        return self.compose(text, [c, cmp_], "pass_percentage",
                            must_include=tuple(required))

    # ---------------------------------------------------------------- Q7
    def branch_placement(self, q):
        category = "circuit" if "circuit" in q.lower() else "core"
        all_rows = self.connector.read(
            "department reports",
            reason="step 1 of 3: read every department row before filtering",
            period=REPORTING_PERIOD)["rows"]
        subset = self.router.call(
            "filter_rows",
            reason=f"step 2 of 3: restrict to {category}-branch departments only",
            rows=all_rows, field="category", value=category)
        assert subset["row_count"] > 0            # explicit non-empty check before computing
        r = self.kpi_chain("KPI-P1",
                           f"step 3 of 3: compute placement over the {category} subset",
                           rows=subset["rows"], scope_label=f"{category} departments")
        c, cmp_ = r["computed"], r["compared"]
        text = (f"Across the {subset['row_count']} {category}-branch departments "
                f"({', '.join(c['departments'])}), placement is "
                f"{_fmt(c['value'], c['unit'])} — computed on the filtered subset "
                f"({', '.join(f'{k}={v}' for k, v in c['inputs_used'].items())}), not on all "
                f"{len(all_rows)} departments. Against the {cmp_['target_source']} target of "
                f"{cmp_['target']:.2f}% that is {cmp_['status']}. Source: {c['source_system']}.")
        return self.compose(text, [c, cmp_, subset, {"total_rows": len(all_rows)}],
                            "branch_placement", must_include=(c["source_system"],))

    # ---------------------------------------------------------------- fallback
    def unrouted(self, q):
        return ("I have no tool path for that question. The tools available to me read "
                "department reports, approved KPI definitions and external benchmarks, and "
                "compute or compare KPIs over them. I am not going to answer it from memory.")

    # ---------------------------------------------------------------- entry
    def answer(self, question: str) -> str:
        route = self.route(question)
        before = len(self.router.trace)
        try:
            text = getattr(self, route)(question)
        except ToolError as exc:
            text = f"I could not answer that: {exc}"
        except AssertionError as exc:
            text = f"Validation gate blocked the answer: {exc}"
        return self.record(question, text, route, before)


assistant = CampusExecutiveAssistant()
print("CampusExecutiveAssistant ready.")
print(f"Routes: {[name for _, name in CampusExecutiveAssistant.ROUTES]} + unrouted fallback")


CampusExecutiveAssistant ready.
Routes: ['placement_drilldown', 'branch_placement', 'placement_vs_target', 'research_vs_peers', 'internship_vs_peers', 'budget_utilisation', 'pass_percentage'] + unrouted fallback


### Cell 9 · Seven questions, seven behaviours

| | Question | What it demonstrates |
|---|---|---|
| Q1 | Placement vs target | A four-tool chain: read → define → compute → compare |
| Q2 | Which departments drag it down | Drill-down evidence, KPI → department → source metric |
| Q3 | Citations vs peers | A second source (benchmark API) joined to our own figure |
| Q4 | Internship vs peers | **Graceful failure** — the benchmark does not exist, and the assistant says so |
| Q5 | Budget utilisation | **Guardrail** — a provisional figure is labelled, never presented as an actual |
| Q6 | Pass percentage vs target | **Guardrail** — a rounded figure would read as on-target; the tool flags it |
| Q7 | Who to recruit as Dean | **No tool path** — the assistant declines instead of guessing |

In [9]:
# =============================================================================
# Cell 9  ·  Six questions, six tool chains
# =============================================================================
# Each question below exercises a different property the lab is graded on.

def show_trace(entries, indent="      "):
    for e in entries:
        flag = "" if e["ok"] else "   <-- returned an error"
        print(f"{indent}{e['step']}. {e['tool']}{flag}")
        print(f"{indent}   why   : {e['reason']}")
        print(f"{indent}   input : {json.dumps(e['input'], default=str)}")
        print(f"{indent}   output: {json.dumps(e['output'], default=str)[:340]}"
              + ("..." if len(json.dumps(e['output'], default=str)) > 340 else ""))


QUESTIONS = [
    ("Q1  tool chaining",
     "What is our institution-wide placement percentage for 2025-26, and how does it "
     "compare to the strategic-plan target?"),
    ("Q2  drill-down evidence",
     "Which departments are dragging the placement percentage below target?"),
    ("Q3  external benchmark available",
     "How do our research citations per faculty compare with peer institutions?"),
    ("Q4  graceful failure — benchmark missing",
     "How does our internship completion compare with peer institutions?"),
    ("Q5  guardrail — provisional figures",
     "What is our budget utilisation this year?"),
    ("Q6  guardrail — a rounded figure would hide the miss",
     "Are we meeting the pass percentage target?"),
    ("Q7  no tool path — the assistant declines rather than guessing",
     "Who should we recruit as Dean of Research next year?"),
]

for label, question in QUESTIONS:
    before = len(assistant.router.trace)
    answer = assistant.answer(question)
    used = assistant.router.trace[before:]
    print("=" * 78)
    print(f"{label}")
    print("=" * 78)
    print(f"  Q: {question}")
    print(f"\n  A: {textwrap.fill(answer, 74, subsequent_indent='     ')}")
    print(f"\n  route selected : {assistant.answers[-1]['route']}")
    print(f"  tools called   : {len(used)}  ->  {' -> '.join(e['tool'] for e in used)}")
    print("  execution log:")
    show_trace(used)
    print()


Q1  tool chaining
  Q: What is our institution-wide placement percentage for 2025-26, and how does it compare to the strategic-plan target?

  A: Institution-wide placement for academic year 2025-26 is 75.12%, computed
     as sum(placed_students) / sum(eligible_students) x 100 over 8
     departments (sum(placed_students)=779, sum(eligible_students)=1037).
     The Strategic Plan 2024-27 target is 80.00%, so we are 4.88
     percentage points below it — status: below target. Source: Placement
     Cell tracker.

  route selected : placement_vs_target
  tools called   : 4  ->  read_department_reports -> read_kpi_definitions -> compute_kpi -> compare_to_target
  execution log:
      1. read_department_reports
         why   : leadership asked how placement stands against target: the per-department metrics live in the reports export
         input : {"period": "academic year 2025-26"}
         output: {"rows": "[8 rows: CSE, ECE, ISE, EEE, MECH, CIVIL, CHEM, BT]", "row_count": 8, "period

### Cell 10 · Assignment 1 — a tool that drafts and cannot send

> *Add a fourth tool that only drafts a message (it must not actually send anything). Extend the assistant so that when a computed value falls below a threshold you define, it calls this tool to draft a notice, and states in the trace why the tool was needed.*

The threshold is a policy constant (`ESCALATION_THRESHOLD_PCT = 5.0`), defined in one place rather than buried in a prompt. The sweep runs over five KPIs, and the tool is called **only** because four of them breached it — the reason string in the trace says exactly that.

Two guardrails come with it, both exercised:

* `draft_notification` has **no send path at all**. "Zero consequential actions without leadership approval" is enforced by construction.
* A finding without a `source` is **refused**, so an uncited figure cannot reach the Principal.

In [10]:
# =============================================================================
# Cell 10 · Assignment 1 — a fourth tool that DRAFTS and never sends
# =============================================================================
# "Add a tool that only drafts a message (it must not actually send anything).
#  Extend the assistant so that when a computed value falls below a threshold
#  you define, it calls this tool to draft a notice, and states in the trace why
#  the tool was needed."
#
# Threshold defined here, in one place, as a policy constant — not buried in a
# prompt. It maps directly to the Chapter 15 rule that agents draft and flag,
# and a named human approves: `draft_notification` structurally cannot send.

ESCALATION_THRESHOLD_PCT = 5.0     # flag a KPI missing its target by > 5% of target
WATCHED_KPIS = ["KPI-A1", "KPI-P1", "KPI-P3", "KPI-R1", "KPI-R2"]


def low_performance_watch(assistant, kpi_ids=WATCHED_KPIS,
                          threshold_pct=ESCALATION_THRESHOLD_PCT):
    rows = assistant.connector.read(
        "department reports",
        reason="performance watch: one read serves every KPI in the sweep",
        period=REPORTING_PERIOD)["rows"]

    scorecard, breaches = [], []
    for kpi_id in kpi_ids:
        r = assistant.kpi_chain(kpi_id, f"performance watch on {kpi_id}", rows=rows)
        c, cmp_ = r["computed"], r["compared"]
        shortfall_pct = max(0.0, -cmp_["variance"] / cmp_["target"] * 100)
        row = {"kpi_id": kpi_id, "name": c["name"], "value": c["value"], "unit": c["unit"],
               "target": cmp_["target"], "status": cmp_["status"],
               "shortfall_pct_of_target": round(shortfall_pct, 2),
               "source": c["source_system"]}
        scorecard.append(row)
        if shortfall_pct > threshold_pct:
            breaches.append(row)

    print(f"  Performance sweep over {len(kpi_ids)} KPIs "
          f"(escalation threshold: shortfall > {threshold_pct}% of target)\n")
    print(f"    {'KPI':<9}{'name':<28}{'value':>10}{'target':>10}{'short by':>11}  status")
    for r in scorecard:
        print(f"    {r['kpi_id']:<9}{r['name']:<28}{r['value']:>10.2f}{r['target']:>10.2f}"
              f"{r['shortfall_pct_of_target']:>10.2f}%  {r['status']}")

    if not breaches:
        print("\n  No KPI breached the threshold — no notification is warranted, "
              "so no notification tool is called.")
        return None

    findings = [{"statement": (f"{b['name']} is {_fmt(b['value'], b['unit'])} against a "
                               f"target of {_fmt(b['target'], b['unit'])} "
                               f"({b['shortfall_pct_of_target']:.2f}% short)"),
                 "source": b["source"]} for b in breaches]

    draft = assistant.router.call(
        "draft_notification",
        reason=(f"{len(breaches)} KPI(s) missed target by more than the "
                f"{threshold_pct}% escalation threshold, so leadership needs a written "
                f"heads-up before the Governing Council cycle — drafting only, because "
                f"circulation requires the Principal's approval"),
        audience="Principal / Registrar",
        subject=f"{len(breaches)} institution KPIs below target for {REPORTING_PERIOD}",
        findings=findings)

    print(f"\n  {len(breaches)} KPI(s) breached the threshold -> draft_notification called.\n")
    print(f"    status                : {draft['status']}")
    print(f"    audience              : {draft['audience']}")
    print(f"    subject               : {draft['subject']}")
    print(f"    findings cited        : {draft['findings_cited']}")
    print(f"    requires approval from: {draft['requires_approval_from']}")
    print(f"    body:\n{textwrap.indent(draft['body'], '      ')}")
    print(f"\n    {draft['note']}")
    return draft


print("=" * 78)
print("ASSIGNMENT 1 · THRESHOLD-TRIGGERED NOTIFICATION (DRAFT ONLY)")
print("=" * 78)
notification = low_performance_watch(assistant)

# ---- The guardrail is real: an unsourced finding is refused ------------------
print("\n" + "-" * 78)
print("GUARDRAIL: every figure in a leadership notification must cite its source")
print("-" * 78)
try:
    assistant.router.call(
        "draft_notification",
        reason="deliberately attempting to draft a notice with an uncited figure",
        audience="Principal / Registrar", subject="Placement update",
        findings=[{"statement": "Placement is around 75% this year"}])   # no 'source'
except ToolError as exc:
    print(f"  refused: {exc}")

print("\n  Note what the tool CANNOT do: there is no send path in its implementation, "
      "\n  so 'zero consequential actions without leadership approval' is enforced by "
      "\n  construction rather than by instruction.")


ASSIGNMENT 1 · THRESHOLD-TRIGGERED NOTIFICATION (DRAFT ONLY)
  Performance sweep over 5 KPIs (escalation threshold: shortfall > 5.0% of target)

    KPI      name                             value    target   short by  status
    KPI-A1   Institution pass percentage      91.85     92.00      0.16%  at risk
    KPI-P1   Placement percentage             75.12     80.00      6.10%  below target
    KPI-P3   Internship completion            73.38     85.00     13.66%  below target
    KPI-R1   Publications per faculty          1.19      1.50     20.94%  below target
    KPI-R2   Citations per faculty             7.26      9.00     19.37%  below target

  4 KPI(s) breached the threshold -> draft_notification called.

    status                : DRAFT - NOT SENT
    audience              : Principal / Registrar
    subject               : 4 institution KPIs below target for academic year 2025-26
    findings cited        : 4
    requires approval from: Principal / Registrar (Chapter 15 human

### Cell 11 · Assignment 2 — read → filter → compute, validated

> *Implement a question that requires three steps: read, filter rows by category, then compute over the filtered subset. Validate the filter step's output (non-empty) before computing.*

The validation is structural: `filter_rows` **refuses to return an empty subset**, and the assistant asserts `row_count > 0` before computing. Both are exercised — including a deliberate filter on a category that does not exist.

The cross-check at the end is the point of the whole lab in one number: the circuit and core subsets recombine exactly to the institution figure when you aggregate the **counts**, and do not when you average the **percentages**.

In [11]:
# =============================================================================
# Cell 11 · Assignment 2 — a three-tool chain with a validated filter step
# =============================================================================
# "Implement a question that requires three steps: read the CSV, filter rows by
#  category, then compute over the filtered subset. Validate the filter step's
#  output (non-empty) before computing."
#
# The validation is not a comment — `filter_rows` refuses to return an empty
# subset at all, and the assistant asserts row_count > 0 before computing. Both
# are exercised below.

print("=" * 78)
print("ASSIGNMENT 2 · READ -> FILTER -> COMPUTE, over a category subset")
print("=" * 78)

for question in ["What is the placement percentage for circuit-branch departments only?",
                 "What is the placement percentage for core-branch departments only?"]:
    before = len(assistant.router.trace)
    answer = assistant.answer(question)
    used = assistant.router.trace[before:]
    print(f"\n  Q: {question}")
    print(f"  A: {textwrap.fill(answer, 74, subsequent_indent='     ')}")
    print(f"\n  chain ({len(used)} calls): {' -> '.join(e['tool'] for e in used)}")
    for e in used:
        if e["tool"] in ("read_department_reports", "filter_rows", "compute_kpi"):
            print(f"      {e['step']}. {e['tool']}")
            print(f"         why   : {e['reason']}")
            print(f"         input : {json.dumps(e['input'], default=str)[:150]}")
            print(f"         output: {json.dumps(e['output'], default=str)[:220]}...")

# ---- The non-empty validation, exercised ------------------------------------
print("\n" + "-" * 78)
print("VALIDATION: the filter step cannot hand an empty subset to the calculator")
print("-" * 78)
rows_all = assistant.connector.read(
    "department reports", reason="fetch rows so the empty-filter case can be demonstrated",
    period=REPORTING_PERIOD)["rows"]
try:
    assistant.router.call("filter_rows",
                          reason="deliberately filtering on a category that does not exist",
                          rows=rows_all, field="category", value="aerospace")
except ToolError as exc:
    print(f"  refused: {exc}")
print("\n  Without that refusal, compute_kpi would have been handed [] and the run would\n"
      "  have produced a division-by-zero or, worse, a plausible-looking 0.00%.")

# ---- Cross-check: the subsets must reconcile to the institution figure -------
print("\n" + "-" * 78)
print("CROSS-CHECK: do the two subsets reconcile with the institution-wide figure?")
print("-" * 78)
# Still through the router — nothing in this notebook calls a tool directly.
circuit = assistant.router.call("compute_kpi", reason="reconciliation: circuit subset",
                                kpi_id="KPI-P1",
                                rows=[r for r in rows_all if r["category"] == "circuit"])
core = assistant.router.call("compute_kpi", reason="reconciliation: core subset",
                             kpi_id="KPI-P1",
                             rows=[r for r in rows_all if r["category"] == "core"])
whole = assistant.router.call("compute_kpi", reason="reconciliation: all departments",
                              kpi_id="KPI-P1", rows=rows_all)
placed = circuit["inputs_used"]["sum(placed_students)"] + core["inputs_used"]["sum(placed_students)"]
eligible = (circuit["inputs_used"]["sum(eligible_students)"]
            + core["inputs_used"]["sum(eligible_students)"])
recombined = placed / eligible * 100
print(f"  circuit     : {circuit['value']:.2f}%  ({circuit['n_departments']} departments)")
print(f"  core        : {core['value']:.2f}%  ({core['n_departments']} departments)")
print(f"  institution : {whole['value']:.2f}%  ({whole['n_departments']} departments)")
print(f"  recombined  : {recombined:.2f}%  <- reconciles with the institution figure: "
      f"{abs(recombined - whole['value']) < 1e-3}")
print(f"\n  Note the plain mean of the two subset percentages would be "
      f"{(circuit['value'] + core['value']) / 2:.2f}% — wrong, because the subsets are "
      f"different sizes.\n  Aggregation happens over the raw counts, never over the "
      f"percentages. That is why compute_kpi\n  takes rows rather than pre-computed values.")


ASSIGNMENT 2 · READ -> FILTER -> COMPUTE, over a category subset

  Q: What is the placement percentage for circuit-branch departments only?
  A: Across the 4 circuit-branch departments (CSE, ECE, ISE, EEE), placement is
     86.35% — computed on the filtered subset (sum(placed_students)=544,
     sum(eligible_students)=630), not on all 8 departments. Against the
     Strategic Plan 2024-27 target of 80.00% that is on target. Source:
     Placement Cell tracker.

  chain (5 calls): read_department_reports -> filter_rows -> read_kpi_definitions -> compute_kpi -> compare_to_target
      46. read_department_reports
         why   : step 1 of 3: read every department row before filtering
         input : {"period": "academic year 2025-26"}
         output: {"rows": "[8 rows: CSE, ECE, ISE, EEE, MECH, CIVIL, CHEM, BT]", "row_count": 8, "period": "academic year 2025-26", "source_system": "Department reports export (Registrar-authorised read)"}...
      47. filter_rows
         why   : step 2

### Cell 12 · Why the tool layer exists

One question, three ways to answer it. The gap between way 2 and way 3 is what "never assert a fact in the prompt that a tool should verify" actually costs — and it is not a rounding error, it is 4.53 percentage points, because averaging department percentages treats a 57-student department as equal to a 228-student one.

Then the same trap on the academic KPI, where rounding alone flips the verdict from *at risk* to *on target*.

In [12]:
# =============================================================================
# Cell 12 · Why the tool layer exists — three answers to one question
# =============================================================================
# "Never assert a fact in the prompt that a tool should verify."  Here is what
# that pitfall actually costs, on this data.

rows = assistant.connector.read("department reports",
                                reason="comparison of three ways to answer one question",
                                period=REPORTING_PERIOD)["rows"]

# --- Way 1: answer from memory (no tool at all) ------------------------------
UNAIDED_SCHEMA = {"type": "object", "properties": {"answer": {"type": "string"}},
                  "required": ["answer"]}


@llm.offline.register("unaided")
def _offline_unaided(payload):
    return {"answer": ("Your institution's placement percentage for 2025-26 is "
                       "approximately 80%, which is broadly in line with the target.")}


unaided = llm.json(task="unaided", system="You are a helpful assistant.",
                   prompt="What is our institution's placement percentage for 2025-26?",
                   schema=UNAIDED_SCHEMA, payload={})["answer"]

# --- Way 2: the arithmetic done the obvious (wrong) way ----------------------
dept_pcts = [r["placed_students"] / r["eligible_students"] * 100 for r in rows]
naive_mean = sum(dept_pcts) / len(dept_pcts)

# --- Way 3: through the tool, on the approved formula ------------------------
grounded = assistant.router.call(
    "compute_kpi", reason="the approved formula aggregates counts, not percentages",
    kpi_id="KPI-P1", rows=rows)

print("=" * 78)
print("ONE QUESTION: what is the institution-wide placement percentage for 2025-26?")
print("=" * 78)
print(f"\n  1. FROM MEMORY, NO TOOL")
print(f"     {textwrap.fill(unaided, 70, subsequent_indent='     ')}")
print(f"     -> unverifiable, uncited, and not derived from any institutional record.")

print(f"\n  2. ARITHMETIC DONE IN THE PROMPT — mean of the 8 department percentages")
print(f"     {' + '.join(f'{p:.2f}' for p in dept_pcts)}")
print(f"     / {len(dept_pcts)}  =  {naive_mean:.2f}%")
print(f"     -> {abs(naive_mean - grounded['value']):.2f} percentage points off. It treats "
      f"BT (57 eligible)\n        and CSE (228 eligible) as equally important.")

print(f"\n  3. THROUGH compute_kpi, ON THE REGISTERED FORMULA")
print(f"     {grounded['formula']}")
print(f"     {' / '.join(str(v) for v in grounded['inputs_used'].values())} x 100  =  "
      f"{grounded['value']:.2f}%")
print(f"     -> cites {grounded['source_system']}, covers "
      f"{grounded['n_departments']} departments, and carries per-department evidence.")

print("\n" + "-" * 78)
print("THE SAME TRAP, ON THE ACADEMIC KPI — where rounding decides the verdict")
print("-" * 78)
pass_kpi = assistant.router.call("compute_kpi", reason="pass percentage, unrounded",
                                 kpi_id="KPI-A1", rows=rows)
pass_cmp = assistant.router.call("compare_to_target", reason="target check on the raw value",
                                 kpi_id="KPI-A1", value=pass_kpi["value"])
print(f"  unrounded : {pass_kpi['value']:.4f}%   target {pass_cmp['target']:.2f}%   "
      f"-> {pass_cmp['status']}")
print(f"  rounded   : {round(pass_kpi['value']):.0f}%        target {pass_cmp['target']:.0f}%"
      f"        -> reads as ON TARGET")
print(f"\n  {textwrap.fill(pass_cmp['rounding_warning'], 74, subsequent_indent='  ')}")
print("\n  Chapter 15 guardrail for the KPI Agent: \"never mask a missed target in a\n"
      "  rounded figure\". compare_to_target enforces it by construction — the warning\n"
      "  is produced by the tool, not remembered by the author of the prompt.")


ONE QUESTION: what is the institution-wide placement percentage for 2025-26?

  1. FROM MEMORY, NO TOOL
     Your institution's placement percentage for 2025-26 is approximately
     80%, which is broadly in line with the target.
     -> unverifiable, uncited, and not derived from any institutional record.

  2. ARITHMETIC DONE IN THE PROMPT — mean of the 8 department percentages
     92.98 + 83.14 + 90.35 + 74.14 + 63.79 + 52.54 + 56.90 + 50.88
     / 8  =  70.59%
     -> 4.53 percentage points off. It treats BT (57 eligible)
        and CSE (228 eligible) as equally important.

  3. THROUGH compute_kpi, ON THE REGISTERED FORMULA
     sum(placed_students) / sum(eligible_students) x 100
     779 / 1037 x 100  =  75.12%
     -> cites Placement Cell tracker, covers 8 departments, and carries per-department evidence.

------------------------------------------------------------------------------
THE SAME TRAP, ON THE ACADEMIC KPI — where rounding decides the verdict
----------------------

### Cell 13 · Real function calling

Lab 2's objective is *function calling*, so here it is: the tool schemas become Gemini `FunctionDeclaration`s and the **model** decides what to call. The loop is driven manually (automatic function calling stays disabled) so every model-chosen call is visible, passes through the same router, and lands in the same audit trail.

**A design decision worth stating.** The graded pipeline in Cells 9–12 uses a deterministic planner, because a rubric needs reproducible traces and Cell 14 asserts exact tool sequences. Model-driven selection is demonstrated here on a coarser tool surface sized for a model — no row payloads travel through the context window. Both routes terminate in the same typed tools, with the same validation and the same grounding check.

Without an API key this cell prints the function declarations and skips the call.

In [13]:
# =============================================================================
# Cell 13 · Real function calling — the model picks the tools
# =============================================================================
# Lab 2's objective is "function calling", so here it is for real: the same tool
# schemas are handed to Gemini as function declarations and the model decides
# what to call. The loop is driven manually (automatic function calling stays
# disabled) so every model-chosen call is visible and still passes through the
# same router, with the same validation and the same audit trail.
#
# Design note, deliberate: the *graded* pipeline in Cells 9-12 uses a
# deterministic planner, because a rubric needs reproducible traces and the
# evaluation suite in Cell 14 asserts exact tool sequences. Model-driven
# selection is demonstrated here, on a coarser tool surface sized for a model
# (no row payloads travel through the context window). Both routes end in the
# same typed tools.

FC_TOOL_SCHEMAS = {
    "list_kpis": {
        "description": "List the approved institution KPI ids, names and domains.",
        "parameters": {"type": "object", "properties": {}, "required": []},
    },
    "get_kpi_value": {
        "description": ("Compute one approved KPI for the current reporting period. "
                        "Optionally restrict to a department category "
                        "('circuit' or 'core')."),
        "parameters": {"type": "object", "properties": {
            "kpi_id": {"type": "string", "description": "e.g. 'KPI-P1'"},
            "category": {"type": "string", "enum": ["circuit", "core"]}},
            "required": ["kpi_id"]},
    },
    "get_kpi_target": {
        "description": "Get the approved target for a KPI and the authority that set it.",
        "parameters": {"type": "object",
                       "properties": {"kpi_id": {"type": "string"}},
                       "required": ["kpi_id"]},
    },
    "get_external_benchmark": {
        "description": ("Get the published peer benchmark for a KPI. Many KPIs have "
                        "none; the call then returns an error and you must say so."),
        "parameters": {"type": "object",
                       "properties": {"kpi_id": {"type": "string"}},
                       "required": ["kpi_id"]},
    },
}


def fc_list_kpis():
    defs = assistant.connector.read("institutional KPIs & dashboards",
                                    reason="model asked for the KPI catalogue")["definitions"]
    return {"kpis": [{"kpi_id": k, "name": v["name"], "domain": v["domain"]}
                     for k, v in defs.items()]}


def fc_get_kpi_value(kpi_id, category=None):
    rows = assistant.connector.read("department reports",
                                    reason="model-selected: value needs the department rows",
                                    period=REPORTING_PERIOD)["rows"]
    if category:
        rows = assistant.router.call("filter_rows",
                                     reason=f"model-selected: restrict to {category} branches",
                                     rows=rows, field="category", value=category)["rows"]
    c = assistant.router.call("compute_kpi", reason="model-selected: compute the KPI",
                              kpi_id=kpi_id, rows=rows)
    return {k: c[k] for k in ("kpi_id", "name", "value", "unit", "formula",
                              "n_departments", "departments", "inputs_used",
                              "source_system", "figure_status")
            if k in c} | ({"caveat": c["caveat"]} if c.get("caveat") else {})


def fc_get_kpi_target(kpi_id):
    d = assistant.connector.read("institutional KPIs & dashboards",
                                 reason="model-selected: fetch the approved target",
                                 kpi_ids=[kpi_id])["definitions"][kpi_id]
    return {"kpi_id": kpi_id, "target": d["target"], "unit": d["unit"],
            "target_source": d["target_source"]}


def fc_get_external_benchmark(kpi_id):
    return assistant.connector.read("external benchmarks",
                                    reason="model-selected: peer benchmark lookup",
                                    required=False, kpi_id=kpi_id)


FC_TOOLS = {"list_kpis": fc_list_kpis, "get_kpi_value": fc_get_kpi_value,
            "get_kpi_target": fc_get_kpi_target,
            "get_external_benchmark": fc_get_external_benchmark}

FC_SYSTEM = """You answer questions for the Principal of an educational institution using
the tools provided. Rules you must not break:
- Never state a figure you did not receive from a tool. Never estimate or round.
- If a tool returns an error, say plainly that the information is unavailable.
- If a figure is marked provisional, say so in your answer.
- Quote the source_system for every figure you report."""


def gemini_function_calling(question: str, max_turns: int = 6):
    """Manual tool-use loop. Returns (answer_text, list_of_model_chosen_calls)."""
    from google.genai import types
    decls = [types.FunctionDeclaration(name=n, description=s["description"],
                                       parameters=s["parameters"])
             for n, s in FC_TOOL_SCHEMAS.items()]
    cfg = types.GenerateContentConfig(
        system_instruction=FC_SYSTEM, temperature=0.0,
        tools=[types.Tool(function_declarations=decls)],
        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True))

    contents = [types.Content(role="user", parts=[types.Part(text=question)])]
    chosen = []
    for turn in range(1, max_turns + 1):
        resp = llm.backend.client.models.generate_content(
            model=llm.backend.model, contents=contents, config=cfg)
        parts = resp.candidates[0].content.parts or []
        calls = [p.function_call for p in parts if getattr(p, "function_call", None)]
        if not calls:
            return (resp.text or "").strip(), chosen
        contents.append(resp.candidates[0].content)
        responses = []
        for fc in calls:
            args = dict(fc.args or {})
            chosen.append({"turn": turn, "tool": fc.name, "args": args})
            try:
                out = FC_TOOLS[fc.name](**args)
            except (ToolError, ToolContractError, KeyError, TypeError) as exc:
                out = {"error": str(exc)}
            responses.append(types.Part.from_function_response(
                name=fc.name, response=_summarise(out)))
        contents.append(types.Content(role="user", parts=responses))
    return "(reached the turn limit without a final answer)", chosen


FC_QUESTION = ("How is placement doing against target this year, and how do we compare "
               "with peer institutions on it?")

print("=" * 78)
print("MODEL-DRIVEN TOOL SELECTION")
print("=" * 78)
if llm.mode != "gemini":
    print("  Skipped: no GEMINI_API_KEY in this session, so there is no model to hand the")
    print("  function declarations to. Add the key and re-run this cell to see Gemini")
    print("  choose the tools itself. The function declarations built above are printed")
    print("  below so the surface is inspectable either way.\n")
    for n, s in FC_TOOL_SCHEMAS.items():
        print(f"    {n}({', '.join(s['parameters']['properties']) or ''})"
              f"  required={s['parameters'].get('required', [])}")
else:
    try:
        print(f"  Q: {FC_QUESTION}\n")
        text, chosen = gemini_function_calling(FC_QUESTION)
        print("  Tools the MODEL chose to call:")
        for c in chosen:
            print(f"    turn {c['turn']}: {c['tool']}({json.dumps(c['args'])})")
        print(f"\n  A: {textwrap.fill(text, 74, subsequent_indent='     ')}")
        outputs = assistant.router.raw_outputs
        ok, bad = ground_check(text, outputs)
        print(f"\n  grounding check on the model's answer: "
              f"{'PASS — every figure traces to a tool output' if ok else f'FAIL — ungrounded: {bad}'}")
    except Exception as exc:
        print(f"  Function-calling demo could not complete: {type(exc).__name__}: {exc}")
        print("  (The deterministic pipeline above is unaffected.)")


MODEL-DRIVEN TOOL SELECTION
  Skipped: no GEMINI_API_KEY in this session, so there is no model to hand the
  function declarations to. Add the key and re-run this cell to see Gemini
  choose the tools itself. The function declarations built above are printed
  below so the surface is inspectable either way.

    list_kpis()  required=[]
    get_kpi_value(kpi_id, category)  required=['kpi_id']
    get_kpi_target(kpi_id)  required=['kpi_id']
    get_external_benchmark(kpi_id)  required=['kpi_id']


### Cell 14 · Evaluation — a regression suite over the tool layer

Four families of test on a fresh assistant: **numeric** (all 9 KPIs against known values, so a changed formula fails loudly), **routing**, **tool sequences** (the right tools in the right order), **error contracts** (every tool's failure path actually fires), and **grounding** (no answer contains a figure a tool did not produce).

In [14]:
# =============================================================================
# Cell 14 · Evaluation — a regression suite over the tool layer
# =============================================================================
# Four families of test, run on a FRESH assistant so each trace is clean:
#
#   1. numeric regression on all 9 KPIs (catches a changed formula)
#   2. routing (the right chain is selected for the right question)
#   3. tool sequences (the right tools, in the right order)
#   4. error contracts (every tool's failure path actually fires)

EXPECTED_KPI_VALUES = {
    "KPI-A1": 91.8519,   # Institution pass percentage [%]
    "KPI-A2": 2.4844,    # CO/PO attainment index [index 0-3]
    "KPI-R1": 1.1859,    # Publications per faculty
    "KPI-R2": 7.2564,    # Citations per faculty
    "KPI-R3": 242.5,     # Grants sanctioned [INR lakh]
    "KPI-P1": 75.1205,   # Placement percentage [%]
    "KPI-P2": 7.667,     # Average package [LPA]
    "KPI-P3": 73.3848,   # Internship completion [%]
    "KPI-F1": 86.9588,   # Budget utilisation [%]  (provisional)
}

EXPECTED_ROUTES = [
    ("What is our placement percentage against the strategic-plan target?", "placement_vs_target"),
    ("Which departments are pulling placement down?",                       "placement_drilldown"),
    ("How do our citations per faculty compare with peer institutions?",    "research_vs_peers"),
    ("How does internship completion compare with peers?",                  "internship_vs_peers"),
    ("What is our budget utilisation?",                                     "budget_utilisation"),
    ("Are we meeting the pass percentage target?",                          "pass_percentage"),
    ("Placement percentage for circuit-branch departments only?",           "branch_placement"),
    ("Who should we hire next year?",                                       "unrouted"),
]

EXPECTED_SEQUENCES = [
    ("What is our placement percentage against the strategic-plan target?",
     ["read_department_reports", "read_kpi_definitions", "compute_kpi", "compare_to_target"]),
    ("Placement percentage for circuit-branch departments only?",
     ["read_department_reports", "filter_rows", "read_kpi_definitions", "compute_kpi",
      "compare_to_target"]),
    ("How does internship completion compare with peers?",
     ["read_department_reports", "read_kpi_definitions", "compute_kpi", "compare_to_target",
      "read_external_benchmark"]),
    ("Who should we hire next year?", []),
]

ERROR_CONTRACT_CASES = [
    ("read_department_reports", dict(period="academic year 2019-20"),
     "no department report export"),
    ("read_kpi_definitions",    dict(kpi_ids=["KPI-Z9"]),             "unknown KPI id(s)"),
    ("read_external_benchmark", dict(kpi_id="KPI-F1"),
     "no published external benchmark"),
    ("compute_kpi",             dict(kpi_id="KPI-P1", rows=[]),       "empty row set"),
    ("compare_to_target",       dict(kpi_id="KPI-Z9", value=1.0),     "unknown KPI id"),
    ("compute_variance",        dict(value=1.0, reference=0.0, label="x"),
     "zero reference"),
    ("filter_rows",             dict(rows=[], field="category", value="circuit"),
     "empty row set"),
    ("draft_notification",      dict(audience="Principal", subject="x", findings=[]),
     "no findings"),
]

results = []


def ev(family, name, passed, detail=""):
    results.append({"family": family, "test": name, "passed": bool(passed), "detail": detail})


# ---- 1. numeric regression --------------------------------------------------
eval_assistant = CampusExecutiveAssistant()
eval_rows = eval_assistant.connector.read(
    "department reports", reason="evaluation suite: one read for the numeric regression",
    period=REPORTING_PERIOD)["rows"]
for kpi_id, expected in EXPECTED_KPI_VALUES.items():
    got = eval_assistant.router.call("compute_kpi", reason=f"regression check {kpi_id}",
                                     kpi_id=kpi_id, rows=eval_rows)
    v = validate_kpi_result(got)
    ev("numeric", f"{kpi_id} == {expected}", abs(got["value"] - expected) < 5e-4
       and v.valid, f"got {got['value']}, validation {'ok' if v.valid else v.failing}")

# ---- 2. routing -------------------------------------------------------------
for question, expected_route in EXPECTED_ROUTES:
    got = eval_assistant.route(question)
    ev("routing", f"{expected_route:<20} <- {question[:44]}", got == expected_route,
       f"got '{got}'")

# ---- 3. tool sequences ------------------------------------------------------
for question, expected_seq in EXPECTED_SEQUENCES:
    probe = CampusExecutiveAssistant()
    probe.answer(question)
    got = probe.router.sequence()
    ev("sequence", f"{question[:52]}", got == expected_seq,
       f"got {got}" if got != expected_seq else f"{len(got)} calls in order")

# ---- 4. error contracts -----------------------------------------------------
for tool, kwargs, fragment in ERROR_CONTRACT_CASES:
    out = TOOLS[tool](**kwargs)
    ev("error-contract", f"{tool} -> error containing '{fragment}'",
       "error" in out and fragment in out["error"], out.get("error", "NO ERROR RETURNED"))

# ---- 5. grounding on every answer produced earlier in this notebook ---------
all_outputs = assistant.router.raw_outputs
for a in assistant.answers:
    ok, bad = ground_check(a["answer"], all_outputs)
    ev("grounding", f"{a['route']:<22} answer has no ungrounded figure", ok,
       f"ungrounded: {bad}" if bad else "every numeric token traces to a tool output")

# ---- report -----------------------------------------------------------------
print("=" * 78)
print("EVALUATION SUITE")
print("=" * 78)
for family in ["numeric", "routing", "sequence", "error-contract", "grounding"]:
    block = [r for r in results if r["family"] == family]
    print(f"\n  {family.upper()}  ({sum(r['passed'] for r in block)}/{len(block)} passed)")
    for r in block:
        print(f"    [{'PASS' if r['passed'] else 'FAIL'}] {r['test']}")
        if not r["passed"]:
            print(f"           {r['detail']}")

failed = [r for r in results if not r["passed"]]
print("\n" + "=" * 78)
print(f"  {len(results) - len(failed)}/{len(results)} evaluation cases passed.")
assert not failed, f"FAILING CASES: {[r['test'] for r in failed]}"


EVALUATION SUITE

  NUMERIC  (9/9 passed)
    [PASS] KPI-A1 == 91.8519
    [PASS] KPI-A2 == 2.4844
    [PASS] KPI-R1 == 1.1859
    [PASS] KPI-R2 == 7.2564
    [PASS] KPI-R3 == 242.5
    [PASS] KPI-P1 == 75.1205
    [PASS] KPI-P2 == 7.667
    [PASS] KPI-P3 == 73.3848
    [PASS] KPI-F1 == 86.9588

  ROUTING  (8/8 passed)
    [PASS] placement_vs_target  <- What is our placement percentage against the
    [PASS] placement_drilldown  <- Which departments are pulling placement down
    [PASS] research_vs_peers    <- How do our citations per faculty compare wit
    [PASS] internship_vs_peers  <- How does internship completion compare with 
    [PASS] budget_utilisation   <- What is our budget utilisation?
    [PASS] pass_percentage      <- Are we meeting the pass percentage target?
    [PASS] branch_placement     <- Placement percentage for circuit-branch depa
    [PASS] unrouted             <- Who should we hire next year?

  SEQUENCE  (4/4 passed)
    [PASS] What is our placement percentage

### Cell 15 · Acceptance checks — the rubric, executed

Twenty-one checks covering the Worklet 15 Lab 2 acceptance line, the instructor checks (*"Are inputs structured? Are tool results used correctly?"*), the Chapter 15 agent guardrails, and both assignments.

In [15]:
# =============================================================================
# Cell 15 · Acceptance checks — the rubric, executed
# =============================================================================
# Worklet 15, Lab 2 acceptance: "Key numbers and lookups always go through
# tools, never hand-done in the prompt."
# Lab guide instructor check: "Are inputs structured? Are tool results used
# correctly?"
#
# Each check passes or raises.

lab2_checks: List[tuple] = []


def bcheck(name, condition, evidence=""):
    lab2_checks.append((name, bool(condition), evidence))


trace = assistant.router.trace
answers = assistant.answers
SOURCE_SYSTEMS = {d["source_system"] for d in KPI_DEFINITIONS.values()} | {
    "Department reports export (Registrar-authorised read)",
    "External benchmark API (NIRF cohort)"}

# --- tool contracts ----------------------------------------------------------
bcheck("B1  every tool declares a typed input schema",
       set(TOOLS) == set(TOOL_SCHEMAS)
       and all(s["parameters"]["type"] == "object" for s in TOOL_SCHEMAS.values()),
       f"{len(TOOLS)} tools, {len(TOOL_SCHEMAS)} schemas")

bcheck("B2  every logged call names a tool that has a schema",
       all(e["tool"] in TOOL_SCHEMAS for e in trace),
       f"{len(trace)} calls, all schema-backed")

_bad_rejected = 0
for _tool, _args in [("compute_kpi", {"kpi_id": "KPI-P1"}),
                     ("compare_to_target", {"kpi_id": "KPI-P1", "value": "75.1"}),
                     ("filter_rows", {"rows": "not-a-list", "field": "category",
                                      "value": "circuit"})]:
    try:
        _probe_router.validate_input(_tool, _args)
    except ToolContractError:
        _bad_rejected += 1
bcheck("B3  malformed arguments are rejected by the schema guard",
       _bad_rejected == 3, f"{_bad_rejected}/3 rejected (missing arg, wrong type, wrong type)")

bcheck("B4  every tool has an exercised error path",
       all(any(r["family"] == "error-contract" and t in r["test"] and r["passed"]
               for r in results)
           for t in ["read_department_reports", "read_kpi_definitions",
                     "read_external_benchmark", "filter_rows", "compute_kpi",
                     "compare_to_target", "compute_variance", "draft_notification"]),
       "all 8 tools returned an explicit {'error': ...} in the evaluation suite")

# --- tool use ---------------------------------------------------------------
bcheck("B5  every tool call is logged with tool, reason, input and output",
       all({"tool", "reason", "input", "output"} <= set(e) and e["reason"] for e in trace),
       f"{len(trace)} logged calls, every one carrying a stated reason")

chain_lengths = [len(a["tools_used"]) for a in answers]
bcheck("B6  tool chaining: at least one question needs a 4+ tool chain",
       max(chain_lengths) >= 4, f"longest chain: {max(chain_lengths)} calls")

three_chain = next((a for a in answers if "filter_rows" in a["tools_used"]), None)
bcheck("B7  Assignment 2: read -> filter -> compute chain exists",
       three_chain is not None
       and three_chain["tools_used"].index("filter_rows")
       < three_chain["tools_used"].index("compute_kpi"),
       f"{' -> '.join(three_chain['tools_used'])}" if three_chain else "not found")

bcheck("B8  the filter step refuses to return an empty subset",
       "error" in filter_rows(rows, "category", "aerospace"),
       filter_rows(rows, "category", "aerospace")["error"])

bcheck("B9  tools are used only when needed",
       next(a for a in answers if a["route"] == "unrouted")["tools_used"] == [],
       "the unroutable question triggered zero tool calls")

# --- results used correctly --------------------------------------------------
bcheck("B10 every computed KPI passed the validation gate (assert v.valid)",
       assistant.validations and all(v.valid for v in assistant.validations),
       f"{len(assistant.validations)} validations, "
       f"{sum(v.valid for v in assistant.validations)} valid")

bcheck("B11 every answer that reports a figure names its source system",
       all(any(src in a["answer"] for src in SOURCE_SYSTEMS)
           for a in answers if a["route"] != "unrouted"),
       f"{sum(1 for a in answers if a['route'] != 'unrouted')} figure-bearing answers checked")

ungrounded_any = []
for a in answers:
    ok, bad = ground_check(a["answer"], assistant.router.raw_outputs)
    ungrounded_any += bad
bcheck("B12 no answer contains a number that a tool did not produce",
       not ungrounded_any, f"ungrounded tokens: {ungrounded_any or 'none'}")

_naive = sum(r["placed_students"] / r["eligible_students"] * 100 for r in rows) / len(rows)
bcheck("B13 the tool result differs from prompt-style arithmetic (so the tool matters)",
       abs(_naive - grounded["value"]) > 1.0,
       f"prompt-style mean {_naive:.2f}% vs tool-computed {grounded['value']:.2f}%")

# --- graceful failure & guardrails -------------------------------------------
_missing = next(a for a in answers if a["route"] == "internship_vs_peers")
bcheck("B14 a missing record is reported honestly, not fabricated",
       "no published external benchmark" in _missing["answer"]
       and "not going to estimate" in _missing["answer"].lower(),
       "the assistant states the benchmark does not exist rather than estimating one")

_budget = next(a for a in answers if a["route"] == "budget_utilisation")
bcheck("B15 provisional figures are labelled, never presented as actuals",
       "PROVISIONAL" in _budget["answer"] and "unreconciled" in _budget["answer"],
       "Finance figure carries its caveat inline")

_pass = next(a for a in answers if a["route"] == "pass_percentage")
bcheck("B16 a rounding that would hide a missed target is flagged",
       "ROUNDING WARNING" in _pass["answer"],
       "91.85% vs a 92.00% target — the tool raises the warning, not the author")

bcheck("B17 the notification tool drafts and cannot send",
       notification["status"] == "DRAFT - NOT SENT"
       and notification["requires_approval_from"].startswith("Principal")
       and not any("send" in name for name in TOOLS),
       "no send path exists in the tool registry")

bcheck("B18 a notification with an uncited figure is refused",
       "error" in draft_notification("Principal", "x", [{"statement": "around 75%"}]),
       draft_notification("Principal", "x", [{"statement": "around 75%"}])["error"])

# --- connector ---------------------------------------------------------------
bcheck("B19 all three sources are reachable through one connector",
       set(Connector.RESOURCES) == {"department reports",
                                    "institutional KPIs & dashboards",
                                    "external benchmarks"},
       f"{sorted(Connector.RESOURCES)}")

try:
    assistant.connector.read("student attendance", reason="probe")
    _conn_guard = False
except ToolContractError:
    _conn_guard = True
bcheck("B20 the connector rejects a resource it does not expose", _conn_guard,
       "read('student attendance') -> ToolContractError")

# --- evaluation --------------------------------------------------------------
bcheck("B21 the evaluation suite passes end to end",
       all(r["passed"] for r in results),
       f"{len(results)}/{len(results)} evaluation cases")

print("=" * 78)
print("LAB 2 ACCEPTANCE CHECKS")
print("=" * 78)
for name, ok, ev_ in lab2_checks:
    print(f"  [{'PASS' if ok else 'FAIL'}]  {name}")
    if ev_:
        print(f"          {ev_}")
_failed = [n for n, ok, _ in lab2_checks if not ok]
print("=" * 78)
print(f"  {len(lab2_checks) - len(_failed)}/{len(lab2_checks)} checks passed.")
assert not _failed, f"FAILED CHECKS: {_failed}"
print("  Lab 2 acceptance criteria satisfied.")


LAB 2 ACCEPTANCE CHECKS
  [PASS]  B1  every tool declares a typed input schema
          8 tools, 8 schemas
  [PASS]  B2  every logged call names a tool that has a schema
          64 calls, all schema-backed
  [PASS]  B3  malformed arguments are rejected by the schema guard
          3/3 rejected (missing arg, wrong type, wrong type)
  [PASS]  B4  every tool has an exercised error path
          all 8 tools returned an explicit {'error': ...} in the evaluation suite
  [PASS]  B5  every tool call is logged with tool, reason, input and output
          64 logged calls, every one carrying a stated reason
  [PASS]  B6  tool chaining: at least one question needs a 4+ tool chain
          longest chain: 6 calls
  [PASS]  B7  Assignment 2: read -> filter -> compute chain exists
          read_department_reports -> filter_rows -> read_kpi_definitions -> compute_kpi -> compare_to_target
  [PASS]  B8  the filter step refuses to return an empty subset
          filter category='aerospace' matche

### Cell 16 · Deliverables

The Lab 2 deliverable is *"tool definitions, sample calls, outputs, and final answer"*. The full nine-KPI scorecard is built first — through the router, like everything else — so the execution log printed below is complete.

In [16]:
# =============================================================================
# Cell 16 · Deliverables — tool definitions, execution log, exported artifacts
# =============================================================================
# The Lab 2 deliverable is "tool definitions, sample calls, outputs, and final
# answer". All four are here, and all four are written out as JSON.

# Build the full nine-KPI scorecard first, through the router like everything
# else, so the execution log printed below is complete rather than partial.
kpi_scorecard = []
for _k in KPI_DEFINITIONS:
    _c = assistant.router.call("compute_kpi", reason=f"executive scorecard: compute {_k}",
                               kpi_id=_k, rows=rows)
    _t = assistant.router.call("compare_to_target",
                               reason=f"executive scorecard: {_k} against its approved target",
                               kpi_id=_k, value=_c["value"])
    _v = validate_kpi_result(_c, _t)
    assert _v.valid, f"scorecard validation failed for {_k}: {_v.failing}"
    assistant.validations.append(_v)
    kpi_scorecard.append({**_c, "target_check": _t})

print("=" * 78)
print("0. INSTITUTION KPI SCORECARD (all 9 approved KPIs, one row each)")
print("=" * 78)
print(f"  {'KPI':<9}{'name':<28}{'value':>12}{'target':>10}{'variance':>11}  status")
for r in kpi_scorecard:
    t = r["target_check"]
    flag = "  [PROVISIONAL]" if r["figure_status"] != "final" else ""
    print(f"  {r['kpi_id']:<9}{r['name']:<28}{r['value']:>12.4f}{t['target']:>10.2f}"
          f"{t['variance']:>+11.4f}  {t['status']}{flag}")
warned = [r for r in kpi_scorecard if r["target_check"]["rounding_warning"]]
print(f"\n  KPIs where a rounded figure would hide a miss: "
      f"{[r['kpi_id'] for r in warned] or 'none'}")

print("\n" + "=" * 78)
print("1. TOOL DEFINITIONS (the schemas handed to the router and to Gemini)")
print("=" * 78)
print(json.dumps(TOOL_SCHEMAS, indent=2)[:3000] + "\n  ... (full set in the JSON export)")

print("\n" + "=" * 78)
print(f"2. FULL EXECUTION LOG — {len(assistant.router.trace)} tool calls")
print("=" * 78)
for e in assistant.router.trace:
    status = "ok" if e["ok"] else "ERROR"
    print(f"\n  {e['step']:>2}. [{status}] {e['tool']}")
    print(f"      why   : {textwrap.fill(e['reason'], 68, subsequent_indent='              ')}")
    print(f"      input : {json.dumps(e['input'], default=str)[:200]}")
    print(f"      output: {json.dumps(e['output'], default=str)[:200]}")

print("\n" + "=" * 78)
print("3. FINAL ANSWERS")
print("=" * 78)
for a in assistant.answers:
    print(f"\n  Q ({a['route']}): {a['question']}")
    print(f"  A: {textwrap.fill(a['answer'], 72, subsequent_indent='     ')}")
    print(f"  tools: {' -> '.join(a['tools_used']) or '(none — no tool was needed)'}")

# ---- 4. export --------------------------------------------------------------
from pathlib import Path
outdir = Path("lab2_artifacts")
outdir.mkdir(exist_ok=True)

lab2_artifacts = {
    "lab2_tool_definitions.json":   TOOL_SCHEMAS,
    "lab2_function_declarations.json": FC_TOOL_SCHEMAS,
    "lab2_execution_trace.json":    assistant.router.trace,
    "lab2_answers.json":            assistant.answers,
    "lab2_kpi_scorecard.json":      kpi_scorecard,
    "lab2_notification_draft.json": notification,
    "lab2_evaluation_results.json": results,
    "lab2_acceptance_checks.json":  [{"check": n, "passed": ok, "evidence": e}
                                     for n, ok, e in lab2_checks],
    "lab2_prompt_log.json":         llm.calls,
}
for fname, obj in lab2_artifacts.items():
    (outdir / fname).write_text(json.dumps(obj, indent=2, ensure_ascii=False, default=str))

print("\n" + "=" * 78)
print("4. ARTIFACTS WRITTEN")
print("=" * 78)
for fname in lab2_artifacts:
    print(f"  {outdir / fname}  ({(outdir / fname).stat().st_size:,} bytes)")

print(f"\n  Backend used for this run: {llm.mode}")
print(f"  Tool calls executed      : {len(assistant.router.trace)}")
print(f"  Validation gates passed  : {sum(v.valid for v in assistant.validations)}"
      f"/{len(assistant.validations)}")
print(f"  Evaluation cases passed  : {sum(r['passed'] for r in results)}/{len(results)}")
print(f"  Acceptance checks passed : {sum(ok for _, ok, _ in lab2_checks)}/{len(lab2_checks)}")


0. INSTITUTION KPI SCORECARD (all 9 approved KPIs, one row each)
  KPI      name                               value    target   variance  status
  KPI-A1   Institution pass percentage      91.8519     92.00    -0.1481  at risk
  KPI-A2   CO/PO attainment index            2.4844      2.50    -0.0156  at risk
  KPI-R1   Publications per faculty          1.1859      1.50    -0.3141  below target
  KPI-R2   Citations per faculty             7.2564      9.00    -1.7436  below target
  KPI-R3   Grants sanctioned               242.5000    300.00   -57.5000  below target
  KPI-P1   Placement percentage             75.1205     80.00    -4.8795  below target
  KPI-P2   Average package                   7.6670      8.00    -0.3330  below target
  KPI-P3   Internship completion            73.3848     85.00   -11.6152  below target
  KPI-F1   Budget utilisation               86.9588     90.00    -3.0412  below target  [PROVISIONAL]

  KPIs where a rounded figure would hide a miss: ['KPI-A1', 'KPI-

---

## Where Labs 1 + 2 leave the worklet

**Nodes activated:** the Coordinator loop (Lab 1) and the tool layer feeding the early data agents (Lab 2).

What the system can now do, end to end:

* refuse to act on an underspecified request, and ask only for what is genuinely missing;
* read its own access scope and plan around provisional data instead of hiding it;
* produce an editable plan that can be revised surgically and audited afterwards;
* reach every source through one connector, with typed, validated tool calls;
* compute institution-wide KPIs from approved formulas, with per-department drill-down evidence;
* label provisional figures, flag a rounding that would hide a missed target, and say "I don't have that" when a record genuinely does not exist;
* draft — never send — a notification for the human gate.

### What the next labs add

| Lab | Adds to this worklet |
|---|---|
| 3 · Skill Creation | The plan and the brief formatter become reusable, schema-typed skills |
| 4 · Memory & Retrieval | Run state separated from the archive of prior briefs |
| 5 · MCP-Style Connector | The `Connector` in Cell 5 becomes a real typed server with auth |
| 6 · Runtime | Budgets, checkpoints and the audit log wrap the whole loop |
| 7–8 · Node graph & swarm | The Academic / Research / Placement / Finance agents run in parallel |
| 9–11 · Grounding, self-healing, tested spec | Validation failure regenerates only the failing KPI |
| 12–14 · Governance & capstone | Human gate, rollback, and the deployed decision brief |

### Reproducing this notebook

* **Offline (no key):** run top to bottom. Every figure above was produced by the tools in this notebook.
* **With Gemini:** add `GEMINI_API_KEY` to Colab Secrets and re-run. The pipeline is unchanged; Cell 13 additionally shows the model selecting tools, and answer wording is model-composed but still grounding-checked.